# GDUT YOLO11n–DCRCF matched ablation · C3

Primary architecture-only comparison on the frozen GDUT split and Stage-1 source. The target arm must pass an in-notebook real-data CUDA smoke before five full seeds.

## Runtime

In [ ]:
import os
from pathlib import Path
import subprocess
import sys

os.environ.update(PYTHONHASHSEED="3407", CUBLAS_WORKSPACE_CONFIG=":4096:8",
                  PYTHONFAULTHANDLER="1", YOLO_CONFIG_DIR="/kaggle/working/yolo_settings")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "ultralytics==8.4.115", "albumentations==2.0.8"])
CODE = Path("/kaggle/working/dcrcf_v4_code")
CODE.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(CODE))
os.environ["PYTHONPATH"] = str(CODE) + os.pathsep + os.environ.get("PYTHONPATH", "")


## Versioned code

The source cells stage importable modules for the subprocess and DDP workers. Preparation source snapshots are evidence references and are never executed here.

In [ ]:
%%writefile /kaggle/working/dcrcf_v4_code/baseline_model.py
"""Importable model construction and DDP trainers for the GDUT baselines."""

import json
import os

from torch import nn
from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import LOGGER, LOCAL_RANK, RANK
from ultralytics.utils.loss import BboxLoss, v8DetectionLoss
from ultralytics.utils.torch_utils import torch_distributed_zero_first

from baseline_modules import ASPP, C2fWithDWR, DWR, ModulatedDCNv2, SBA, nwd_loss_pixels
from plain_logging import configure_plain_logging

configure_plain_logging()
REVISION = "gdut_paper_protocol_v2"
BASELINES = {"yolo11_dcrcf": "yolo11n", "yolov8_dwr_aspp_nwd": "yolov8s"}


class CIoUNWDBboxLoss(BboxLoss):
    def __init__(self, reg_max, constant=12.8, nwd_weight=0.5):
        super().__init__(reg_max)
        if not 0 <= nwd_weight <= 1 or constant <= 0:
            raise ValueError("Invalid NWD mixture or normalization constant.")
        self.constant = constant
        self.nwd_weight = nwd_weight

    def forward(
        self,
        pred_dist,
        pred_bboxes,
        anchor_points,
        target_bboxes,
        target_scores,
        target_scores_sum,
        fg_mask,
        imgsz,
        stride,
    ):
        ciou, dfl = super().forward(
            pred_dist,
            pred_bboxes,
            anchor_points,
            target_bboxes,
            target_scores,
            target_scores_sum,
            fg_mask,
            imgsz,
            stride,
        )
        # Convert feature-grid box coordinates to input-image pixels.
        pixel_scale = stride.unsqueeze(0).expand(pred_bboxes.shape[0], -1, 4)[fg_mask]
        pred_pixels = pred_bboxes[fg_mask].float() * pixel_scale
        target_pixels = target_bboxes[fg_mask].float() * pixel_scale
        quality = target_scores[fg_mask].sum(-1, keepdim=True)
        nwd = (
            nwd_loss_pixels(pred_pixels, target_pixels, self.constant) * quality
        ).sum()
        nwd = nwd / target_scores_sum
        return (1 - self.nwd_weight) * ciou + self.nwd_weight * nwd, dfl


class PaperBaselineModel(DetectionModel):
    def info(self, detailed=False, verbose=True, imgsz=640):
        """Return layer, parameter and gradient counts."""
        if not verbose:
            return
        layers = sum(not module._modules for module in self.modules())
        parameters = sum(parameter.numel() for parameter in self.parameters())
        gradients = sum(p.numel() for p in self.parameters() if p.requires_grad)
        LOGGER.info(
            f"{self.baseline_id}: {layers} layers, {parameters:,} parameters, "
            f"{gradients:,} gradients; FLOPs not profiled"
        )
        return layers, parameters, gradients, 0.0

    def init_criterion(self):
        criterion = v8DetectionLoss(self)
        if self.baseline_id == "yolov8_dwr_aspp_nwd":
            criterion.bbox_loss = CIoUNWDBboxLoss(
                self.model[-1].reg_max, self.nwd_constant, self.nwd_weight
            ).to(next(self.parameters()).device)
        return criterion


def graph_layer(module, index, inputs=-1):
    module.i, module.f = index, inputs
    module.type = type(module).__name__
    module.np = sum(parameter.numel() for parameter in module.parameters())
    return module


def module_inventory(model):
    classes = (DWR, ASPP, ModulatedDCNv2, SBA)
    return {
        cls.__name__: [
            name for name, module in model.named_modules() if isinstance(module, cls)
        ]
        for cls in classes
    }


def resize_neck_input(block, channels):
    """Adapt the input projection after replacing concat with an SBA output."""
    previous = block.cv1
    block.cv1 = type(previous)(channels, previous.conv.out_channels, 1, 1)
    return block


def apply_baseline(model, baseline_id, nwd_constant=12.8, nwd_weight=0.5):
    if hasattr(model, "baseline_id") or hasattr(model, "research_variant"):
        raise ValueError("The baseline graph has already been constructed.")
    if baseline_id not in BASELINES:
        raise ValueError(baseline_id)
    old = list(model.model)
    if baseline_id == "yolov8_dwr_aspp_nwd":
        if len(old) != 23 or old[9].__class__.__name__ != "SPPF":
            raise ValueError("Expected the standard YOLOv8s graph.")
        for index in (2, 4, 6):
            model.model[index] = graph_layer(
                C2fWithDWR(old[index]), index, old[index].f
            )
        model.model[9] = graph_layer(
            ASPP(old[9].cv1.conv.in_channels, old[9].cv2.conv.out_channels), 9
        )
    else:
        if len(old) != 24 or old[10].__class__.__name__ != "C2PSA":
            raise ValueError("Expected the standard YOLO11 P3/P4/P5 graph.")
        channels = {i: old[i].cv2.conv.out_channels for i in (4, 6, 8)}
        for index in (6, 8):
            old[index] = graph_layer(
                ModulatedDCNv2(old[index].cv1.conv.in_channels, channels[index]), index
            )
        p3, p4, p5 = channels[4], channels[6], channels[8]
        n3 = old[16].cv2.conv.out_channels
        n4 = old[13].cv2.conv.out_channels
        n5 = old[22].cv2.conv.out_channels
        layers = old[:11] + [
            graph_layer(SBA(p5, p4, n4), 11, [10, 6]),
            graph_layer(resize_neck_input(old[13], n4), 12),
            graph_layer(SBA(n4, p3, n3), 13, [12, 4]),
            graph_layer(resize_neck_input(old[16], n3), 14),
            graph_layer(SBA(n3, n4, n4), 15, [14, 12]),
            graph_layer(resize_neck_input(old[19], n4), 16),
            graph_layer(SBA(n4, p5, n5), 17, [16, 10]),
            graph_layer(resize_neck_input(old[22], n5), 18),
            graph_layer(old[23], 19, [14, 16, 18]),
        ]
        model.model = nn.Sequential(*layers)
        model.save = sorted(
            {
                j
                for layer in layers
                for j in ([layer.f] if isinstance(layer.f, int) else layer.f)
                if j != -1
            }
        )
    model.__class__ = PaperBaselineModel
    model.baseline_id = baseline_id
    model.baseline_revision = REVISION
    model.nwd_constant, model.nwd_weight = nwd_constant, nwd_weight
    model.reconstruction_status = "paper_based_reimplementation_not_author_code"
    assert_inventory(model)
    return model


def assert_inventory(model):
    expected = {
        "yolo11_dcrcf": {"DWR": 0, "ASPP": 0, "ModulatedDCNv2": 2, "SBA": 4},
        "yolov8_dwr_aspp_nwd": {"DWR": 3, "ASPP": 1, "ModulatedDCNv2": 0, "SBA": 0},
    }[model.baseline_id]
    actual = {key: len(value) for key, value in module_inventory(model).items()}
    if actual != expected:
        raise RuntimeError(f"Architecture mismatch: {actual} != {expected}")
    if list(model.stride.cpu().tolist()) != [8, 16, 32]:
        raise RuntimeError("Detection strides must remain 8/16/32.")


class PaperBaselineTrainer(DetectionTrainer):
    baseline_id = "yolo11_dcrcf"

    def final_eval(self):
        """Finalize checkpoint files; the launcher validates them after DDP exits."""
        from ultralytics.engine.trainer import strip_optimizer

        with torch_distributed_zero_first(LOCAL_RANK):
            if RANK in {-1, 0}:
                latest = strip_optimizer(self.last) if self.last.exists() else {}
                if not self.best.exists():
                    raise FileNotFoundError(
                        f"No best checkpoint was saved: {self.best}"
                    )
                strip_optimizer(
                    self.best, updates={"train_results": latest.get("train_results")}
                )
                marker = dict(
                    baseline_id=self.baseline_id,
                    revision=REVISION,
                    epochs_completed=self.epoch + 1,
                    best_epoch=getattr(self.stopper, "best_epoch", None),
                    final_validation="deferred_to_single_process_fp32",
                )
                (self.save_dir / "training_complete.json").write_text(
                    json.dumps(marker, indent=2), encoding="utf-8"
                )
                LOGGER.info("Checkpoints saved. Final validation will run outside DDP.")

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = self.set_model_names_for_load(
            DetectionModel(
                cfg, nc=self.data["nc"], ch=self.data["channels"], verbose=verbose
            )
        )
        if weights is not None:
            raise ValueError(
                "This protocol trains a new baseline from YAML without Stage1 weights."
            )
        options = json.loads(os.environ.get("GDUT_BASELINE_OPTIONS", "{}"))
        apply_baseline(model, self.baseline_id, **options)
        report = {
            "baseline_id": self.baseline_id,
            "revision": REVISION,
            "rank": os.environ.get("RANK", "-1"),
            "nc": self.data["nc"],
            "modules": module_inventory(model),
            "strides": model.stride.tolist(),
            "nwd_constant": model.nwd_constant,
            "nwd_weight": model.nwd_weight,
            "box_loss": "CIoU+NWD" if self.baseline_id.startswith("yolov8") else "CIoU",
            "parameters": sum(p.numel() for p in model.parameters()),
        }
        self.save_dir.mkdir(parents=True, exist_ok=True)
        (self.save_dir / f"architecture_rank_{report['rank']}.json").write_text(
            json.dumps(report, indent=2), encoding="utf-8"
        )
        print(json.dumps(report, indent=2), flush=True)
        return model


class DCRCFTrainer(PaperBaselineTrainer):
    baseline_id = "yolo11_dcrcf"


class DWRASPPTrainer(PaperBaselineTrainer):
    baseline_id = "yolov8_dwr_aspp_nwd"


TRAINERS = {"yolo11_dcrcf": DCRCFTrainer, "yolov8_dwr_aspp_nwd": DWRASPPTrainer}


In [ ]:
%%writefile /kaggle/working/dcrcf_v4_code/baseline_modules.py
"""DCNv2, selective boundary aggregation, DWR and ASPP modules."""

import torch
from torch import nn
from torch.nn import functional as F
from torchvision.ops import deform_conv2d


class ConvBNReLU(nn.Sequential):
    def __init__(self, c1, c2, kernel=1, dilation=1, groups=1):
        padding = dilation * (kernel - 1) // 2
        super().__init__(
            nn.Conv2d(
                c1,
                c2,
                kernel,
                padding=padding,
                dilation=dilation,
                groups=groups,
                bias=False,
            ),
            nn.BatchNorm2d(c2),
            nn.ReLU(inplace=False),
        )


class DWR(nn.Module):
    """Song et al., Figure 2: 2c -> 3c -> (2c,c,c) -> 2c, residual."""

    def __init__(self, channels):
        super().__init__()
        if channels % 2:
            raise ValueError("DWR requires an even channel count.")
        half = channels // 2
        self.region = ConvBNReLU(channels, 3 * half, kernel=3)
        self.branches = nn.ModuleList(
            [
                nn.Conv2d(half, channels, 3, padding=1, groups=half, bias=False),
                nn.Conv2d(
                    half, half, 3, padding=3, dilation=3, groups=half, bias=False
                ),
                nn.Conv2d(
                    half, half, 3, padding=5, dilation=5, groups=half, bias=False
                ),
            ]
        )
        self.norm = nn.BatchNorm2d(2 * channels)
        self.project = nn.Conv2d(2 * channels, channels, 1, bias=False)

    def forward(self, x):
        groups = self.region(x).chunk(3, dim=1)
        features = torch.cat([branch(t) for branch, t in zip(self.branches, groups)], 1)
        return x + self.project(self.norm(features))


class C2fWithDWR(nn.Module):
    def __init__(self, c2f):
        super().__init__()
        self.c2f = c2f
        self.dwr = DWR(c2f.cv2.conv.out_channels)

    def forward(self, x):
        return self.dwr(self.c2f(x))


class ASPP(nn.Module):
    """Song et al., Figure 3: rates 6/12/18 and image-level pooling."""

    def __init__(self, c1, c2, hidden=256):
        super().__init__()
        self.branches = nn.ModuleList(
            [
                ConvBNReLU(c1, hidden),
                ConvBNReLU(c1, hidden, kernel=3, dilation=6),
                ConvBNReLU(c1, hidden, kernel=3, dilation=12),
                ConvBNReLU(c1, hidden, kernel=3, dilation=18),
            ]
        )
        # Global pooling branch: 1x1 convolution followed by ReLU.
        self.pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(c1, hidden, 1, bias=True),
            nn.ReLU(inplace=False),
        )
        self.project = ConvBNReLU(5 * hidden, c2)

    def forward(self, x):
        pooled = F.interpolate(
            self.pool(x), size=x.shape[-2:], mode="bilinear", align_corners=False
        )
        return self.project(
            torch.cat([branch(x) for branch in self.branches] + [pooled], 1)
        )


class ModulatedDCNv2(nn.Module):
    """A modulated 3x3 deformable convolution with learned offsets and masks."""

    def __init__(self, c1, c2):
        super().__init__()
        self.offset_mask = nn.Conv2d(c1, 27, 3, padding=1)
        self.weight = nn.Parameter(torch.empty(c2, c1, 3, 3))
        self.norm = nn.BatchNorm2d(c2)
        self.act = nn.SiLU()
        nn.init.kaiming_uniform_(self.weight, a=5**0.5)
        nn.init.zeros_(self.offset_mask.weight)
        nn.init.zeros_(self.offset_mask.bias)

    def forward(self, x):
        original_dtype = x.dtype
        # Run the compiled deformable operator in float32 under AMP and FP16 validation.
        with torch.autocast(device_type=x.device.type, enabled=False):
            features = x.float()
            fields = F.conv2d(
                features,
                self.offset_mask.weight.float(),
                self.offset_mask.bias.float(),
                padding=1,
            )
            offset, mask_logits = fields[:, :18], fields[:, 18:]
            result = deform_conv2d(
                features,
                offset,
                self.weight.float(),
                padding=(1, 1),
                mask=mask_logits.sigmoid(),
            )
        return self.act(self.norm(result.to(original_dtype)))


class RecalibrationUnit(nn.Module):
    """Recalibrate two feature tensors with 3x3 sigmoid gates."""

    def __init__(self, channels):
        super().__init__()
        self.gate_a = nn.Conv2d(channels, channels, 3, padding=1)
        self.gate_b = nn.Conv2d(channels, channels, 3, padding=1)

    def forward(self, a, b):
        ga, gb = self.gate_a(a).sigmoid(), self.gate_b(b).sigmoid()
        return a * ga + (1.0 - ga) * b * gb


class SBA(nn.Module):
    """Two RAUs and a 1x1 projection; output resolution follows the second input."""

    def __init__(self, c1, c2, out_channels):
        super().__init__()
        self.first = nn.Conv2d(c1, out_channels, 1)
        self.second = nn.Conv2d(c2, out_channels, 1)
        self.rau_ab = RecalibrationUnit(out_channels)
        self.rau_ba = RecalibrationUnit(out_channels)
        self.project = nn.Conv2d(2 * out_channels, out_channels, 1)

    def forward(self, inputs):
        first, second = inputs
        a, b = self.first(first), self.second(second)
        a = F.interpolate(a, size=b.shape[-2:], mode="bilinear", align_corners=False)
        return self.project(torch.cat([self.rau_ab(a, b), self.rau_ba(b, a)], 1))


def nwd_loss_pixels(predicted, target, constant):
    """Gaussian Wasserstein box loss; xyxy coordinates and C use input-image pixels."""
    if constant <= 0:
        raise ValueError("NWD normalization constant must be positive.")
    pred_center = (predicted[..., :2] + predicted[..., 2:]) / 2
    true_center = (target[..., :2] + target[..., 2:]) / 2
    pred_half_size = (predicted[..., 2:] - predicted[..., :2]) / 2
    true_half_size = (target[..., 2:] - target[..., :2]) / 2
    delta = torch.cat([pred_center - true_center, pred_half_size - true_half_size], -1)
    distance = torch.linalg.vector_norm(delta.float(), dim=-1, keepdim=True)
    return -torch.expm1(-distance / constant)


In [ ]:
%%writefile /kaggle/working/dcrcf_v4_code/custom_ablation_modules.py
"""CoordConv, RepConv, routed attention, and Focal-EIoU modules."""

from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F


def autopad(k: int, p: int | None = None, d: int = 1) -> int:
    """Return padding that preserves spatial shape for odd kernels."""
    if p is None:
        p = d * (k - 1) // 2
    return p


class ConvBNAct(nn.Module):
    """Small Conv-BN-SiLU block compatible with YOLO-style modules."""

    def __init__(
        self,
        c1: int,
        c2: int,
        k: int = 1,
        s: int = 1,
        p: int | None = None,
        g: int = 1,
        d: int = 1,
        act: bool = True,
    ) -> None:
        super().__init__()
        self.conv = nn.Conv2d(
            c1, c2, k, s, autopad(k, p, d), groups=g, dilation=d, bias=False
        )
        self.bn = nn.BatchNorm2d(c2)
        self.act = nn.SiLU(inplace=True) if act else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.act(self.bn(self.conv(x)))


class CoordConv(nn.Module):
    """
    CoordConv layer: append normalized x/y coordinate channels before a conv.

    """

    def __init__(
        self, c1: int, c2: int, k: int = 3, s: int = 1, with_r: bool = False
    ) -> None:
        super().__init__()
        self.with_r = with_r
        extra = 3 if with_r else 2
        self.conv = ConvBNAct(c1 + extra, c2, k=k, s=s)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, _, h, w = x.shape
        yy = (
            torch.linspace(-1.0, 1.0, h, device=x.device, dtype=x.dtype)
            .view(1, 1, h, 1)
            .expand(b, 1, h, w)
        )
        xx = (
            torch.linspace(-1.0, 1.0, w, device=x.device, dtype=x.dtype)
            .view(1, 1, 1, w)
            .expand(b, 1, h, w)
        )
        coords = [xx, yy]
        if self.with_r:
            rr = torch.sqrt(torch.clamp(xx.square() + yy.square(), min=0.0))
            coords.append(rr)
        return self.conv(torch.cat([x, *coords], dim=1))


class RepConv(nn.Module):
    """
    RepConv block with 3x3, 1x1, and optional identity branches during training.

    Call `switch_to_deploy()` before ONNX/TensorRT export. The deploy path is a
    single 3x3 Conv2d plus activation.
    """

    def __init__(
        self,
        c1: int,
        c2: int,
        k: int = 3,
        s: int = 1,
        deploy: bool = False,
        act: bool = True,
    ) -> None:
        super().__init__()
        assert k == 3, "This RepConv implementation fuses to a 3x3 kernel."
        self.deploy = deploy
        self.in_channels = c1
        self.out_channels = c2
        self.stride = s
        self.act = nn.SiLU(inplace=True) if act else nn.Identity()

        if deploy:
            self.rbr_reparam = nn.Conv2d(c1, c2, 3, s, 1, bias=True)
        else:
            self.rbr_dense = nn.Sequential(
                nn.Conv2d(c1, c2, 3, s, 1, bias=False),
                nn.BatchNorm2d(c2),
            )
            self.rbr_1x1 = nn.Sequential(
                nn.Conv2d(c1, c2, 1, s, 0, bias=False),
                nn.BatchNorm2d(c2),
            )
            self.rbr_identity = nn.BatchNorm2d(c1) if c1 == c2 and s == 1 else None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.deploy:
            return self.act(self.rbr_reparam(x))
        out = self.rbr_dense(x) + self.rbr_1x1(x)
        if self.rbr_identity is not None:
            out = out + self.rbr_identity(x)
        return self.act(out)

    @staticmethod
    def _fuse_conv_bn(
        branch: nn.Sequential | nn.BatchNorm2d, channels: int
    ) -> tuple[torch.Tensor, torch.Tensor]:
        if isinstance(branch, nn.Sequential):
            conv = branch[0]
            bn = branch[1]
            kernel = conv.weight
        else:
            bn = branch
            input_dim = channels
            kernel = torch.zeros(
                (input_dim, input_dim, 3, 3),
                device=bn.weight.device,
                dtype=bn.weight.dtype,
            )
            for i in range(input_dim):
                kernel[i, i, 1, 1] = 1.0

        std = torch.sqrt(bn.running_var + bn.eps)
        t = (bn.weight / std).reshape(-1, 1, 1, 1)
        fused_kernel = kernel * t
        fused_bias = bn.bias - bn.running_mean * bn.weight / std
        return fused_kernel, fused_bias

    @staticmethod
    def _pad_1x1_to_3x3(kernel: torch.Tensor) -> torch.Tensor:
        if kernel.size(2) == 3:
            return kernel
        return F.pad(kernel, [1, 1, 1, 1])

    def get_equivalent_kernel_bias(self) -> tuple[torch.Tensor, torch.Tensor]:
        k3, b3 = self._fuse_conv_bn(self.rbr_dense, self.out_channels)
        k1, b1 = self._fuse_conv_bn(self.rbr_1x1, self.out_channels)
        if self.rbr_identity is None:
            kid = torch.zeros_like(k3)
            bid = torch.zeros_like(b3)
        else:
            kid, bid = self._fuse_conv_bn(self.rbr_identity, self.out_channels)
        return k3 + self._pad_1x1_to_3x3(k1) + kid.to(k3.device), b3 + b1 + bid.to(
            b3.device
        )

    def switch_to_deploy(self) -> None:
        if self.deploy:
            return
        kernel, bias = self.get_equivalent_kernel_bias()
        self.rbr_reparam = nn.Conv2d(
            self.in_channels,
            self.out_channels,
            3,
            self.stride,
            1,
            bias=True,
        )
        self.rbr_reparam.weight.data = kernel.detach().clone()
        self.rbr_reparam.bias.data = bias.detach().clone()
        del self.rbr_dense
        del self.rbr_1x1
        if hasattr(self, "rbr_identity"):
            del self.rbr_identity
        self.deploy = True


class BiFormerBlockLite(nn.Module):
    """
    Region routing and fine token attention with a residual connection.

    Region-level routing selects the key/value tokens used by fine attention.
    The residual block uses four heads and eight-pixel regions by default.
    """

    def __init__(
        self, channels: int, num_heads: int = 4, region_size: int = 8, topk: int = 4
    ) -> None:
        super().__init__()
        assert channels % num_heads == 0, "channels must be divisible by num_heads"
        self.channels = channels
        self.num_heads = num_heads
        self.region_size = region_size
        self.topk = topk
        self.qkv = nn.Conv2d(channels, channels * 3, 1, bias=False)
        self.proj = nn.Conv2d(channels, channels, 1, bias=False)
        self.norm = nn.BatchNorm2d(channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, h, w = x.shape
        rs = self.region_size
        pad_h = (rs - h % rs) % rs
        pad_w = (rs - w % rs) % rs
        x_pad = F.pad(x, (0, pad_w, 0, pad_h))
        hp, wp = x_pad.shape[-2:]
        gh, gw = hp // rs, wp // rs

        q, k, v = self.qkv(x_pad).chunk(3, dim=1)
        q_regions = q.unfold(2, rs, rs).unfold(3, rs, rs).contiguous()
        k_regions = k.unfold(2, rs, rs).unfold(3, rs, rs).contiguous()
        v_regions = v.unfold(2, rs, rs).unfold(3, rs, rs).contiguous()

        # [B, C, GH, GW, RS, RS] -> [B, R, T, C]
        q_tokens = q_regions.permute(0, 2, 3, 4, 5, 1).reshape(b, gh * gw, rs * rs, c)
        k_tokens = k_regions.permute(0, 2, 3, 4, 5, 1).reshape(b, gh * gw, rs * rs, c)
        v_tokens = v_regions.permute(0, 2, 3, 4, 5, 1).reshape(b, gh * gw, rs * rs, c)

        q_region = q_tokens.mean(dim=2)
        k_region = k_tokens.mean(dim=2)
        route_logits = torch.matmul(q_region, k_region.transpose(-1, -2)) / (c**0.5)
        topk = min(self.topk, gh * gw)
        route_idx = route_logits.topk(topk, dim=-1).indices

        out_regions = []
        head_dim = c // self.num_heads
        for region_idx in range(gh * gw):
            selected = route_idx[:, region_idx]
            k_sel = torch.stack(
                [
                    k_tokens[bi, selected[bi]].reshape(topk * rs * rs, c)
                    for bi in range(b)
                ],
                dim=0,
            )
            v_sel = torch.stack(
                [
                    v_tokens[bi, selected[bi]].reshape(topk * rs * rs, c)
                    for bi in range(b)
                ],
                dim=0,
            )
            q_cur = q_tokens[:, region_idx]

            qh = q_cur.reshape(b, rs * rs, self.num_heads, head_dim).transpose(1, 2)
            kh = k_sel.reshape(b, topk * rs * rs, self.num_heads, head_dim).transpose(
                1, 2
            )
            vh = v_sel.reshape(b, topk * rs * rs, self.num_heads, head_dim).transpose(
                1, 2
            )
            attn = torch.softmax(
                torch.matmul(qh, kh.transpose(-1, -2)) / (head_dim**0.5), dim=-1
            )
            out = torch.matmul(attn, vh).transpose(1, 2).reshape(b, rs * rs, c)
            out_regions.append(out)

        y = torch.stack(out_regions, dim=1).reshape(b, gh, gw, rs, rs, c)
        y = y.permute(0, 5, 1, 3, 2, 4).reshape(b, c, hp, wp)
        y = y[:, :, :h, :w]
        return x + self.norm(self.proj(y))


def xywh_to_xyxy(boxes: torch.Tensor) -> torch.Tensor:
    x, y, w, h = boxes.unbind(-1)
    half_w = w / 2
    half_h = h / 2
    return torch.stack((x - half_w, y - half_h, x + half_w, y + half_h), dim=-1)


def focal_eiou_loss(
    pred_boxes: torch.Tensor,
    target_boxes: torch.Tensor,
    xywh: bool = True,
    gamma: float = 0.5,
    eps: float = 1e-7,
    reduction: str = "mean",
) -> torch.Tensor:
    """
    Focal-EIoU box regression loss.

    Args:
        pred_boxes: [N, 4] predicted boxes.
        target_boxes: [N, 4] target boxes.
        xywh: True when boxes are center-x, center-y, width, height.
        gamma: focal weighting exponent.
    """
    if xywh:
        pred = xywh_to_xyxy(pred_boxes)
        target = xywh_to_xyxy(target_boxes)
    else:
        pred = pred_boxes
        target = target_boxes

    px1, py1, px2, py2 = pred.unbind(-1)
    tx1, ty1, tx2, ty2 = target.unbind(-1)
    pw = (px2 - px1).clamp(min=eps)
    ph = (py2 - py1).clamp(min=eps)
    tw = (tx2 - tx1).clamp(min=eps)
    th = (ty2 - ty1).clamp(min=eps)

    inter_x1 = torch.maximum(px1, tx1)
    inter_y1 = torch.maximum(py1, ty1)
    inter_x2 = torch.minimum(px2, tx2)
    inter_y2 = torch.minimum(py2, ty2)
    inter = (inter_x2 - inter_x1).clamp(min=0) * (inter_y2 - inter_y1).clamp(min=0)
    union = pw * ph + tw * th - inter + eps
    iou = (inter / union).clamp(min=eps, max=1.0)

    pcx = (px1 + px2) / 2
    pcy = (py1 + py2) / 2
    tcx = (tx1 + tx2) / 2
    tcy = (ty1 + ty2) / 2
    center_dist = (pcx - tcx).square() + (pcy - tcy).square()

    cw = (torch.maximum(px2, tx2) - torch.minimum(px1, tx1)).clamp(min=eps)
    ch = (torch.maximum(py2, ty2) - torch.minimum(py1, ty1)).clamp(min=eps)
    c2 = cw.square() + ch.square() + eps

    eiou = (
        1.0
        - iou
        + center_dist / c2
        + (pw - tw).square() / (cw.square() + eps)
        + (ph - th).square() / (ch.square() + eps)
    )
    loss = iou.pow(gamma) * eiou

    if reduction == "mean":
        return loss.mean()
    if reduction == "sum":
        return loss.sum()
    if reduction == "none":
        return loss
    raise ValueError(f"Unsupported reduction: {reduction}")


In [ ]:
%%writefile /kaggle/working/dcrcf_v4_code/gdut_data.py
"""Load the existing GDUT split without generating or augmenting samples."""

import hashlib
import json
from pathlib import Path

import yaml

NAMES = ["blue", "white", "yellow", "red", "none"]


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def prepared_dataset(explicit=""):
    candidates = (
        [Path(explicit)]
        if explicit
        else [p.parent for p in Path("/kaggle/input").rglob("dataset_manifest.json")]
    )
    found = []
    for root in candidates:
        path = root / "dataset_manifest.json"
        if not path.is_file():
            continue
        meta = json.loads(path.read_text(encoding="utf-8"))
        if meta.get("dataset") == "gdut":
            found.append((root, meta, sha256(path)))
    if len(found) != 1:
        raise ValueError("Mount one GDUT preparation output or set PREPARED_DIR.")
    root, meta, fingerprint = found[0]
    if meta["names"] != NAMES:
        raise ValueError(
            "Expected the GDUT class order: blue, white, yellow, red, none."
        )
    if (
        meta.get("preparation_version", 1) >= 2
        and not (root / "preparation_complete.json").exists()
    ):
        raise ValueError("The preparation output is incomplete.")
    for split in ("train", "val", "test"):
        for kind in ("images", "labels"):
            if not (root / "base" / kind / split).is_dir():
                raise FileNotFoundError(root / "base" / kind / split)
    return root, meta, fingerprint


def data_yaml(root, output_directory):
    output_directory = Path(output_directory)
    output_directory.mkdir(parents=True, exist_ok=True)
    config = dict(
        path=str(root),
        names=dict(enumerate(NAMES)),
        train="base/images/train",
        val="base/images/val",
        test="base/images/test",
    )
    target = output_directory / "gdut.yaml"
    target.write_text(yaml.safe_dump(config), encoding="utf-8")
    return str(target)


In [ ]:
%%writefile /kaggle/working/dcrcf_v4_code/plain_logging.py
"""Plain-text formatting for Ultralytics console messages."""

import logging
import re


ANSI_ESCAPE = re.compile(r"\x1b\[[0-?]*[ -/]*[@-~]")
DECORATIONS = re.compile(r"[\U0001f000-\U0001faff\u2600-\u27bf\ufe0f\u200d]")
REPLACEMENTS = str.maketrans(
    {
        "\u2192": "->",
        "\u2190": "<-",
        "\u2194": "<->",
        "\u2265": ">=",
        "\u2264": "<=",
        "\u00d7": "x",
        "\u2705": "[OK]",
        "\u2713": "[OK]",
        "\u274c": "[FAIL]",
        "\u2717": "[FAIL]",
    }
)


def plain_text(message):
    """Remove console decoration while preserving labels and numeric values."""
    message = ANSI_ESCAPE.sub("", message).translate(REPLACEMENTS)
    return DECORATIONS.sub("", message)


class PlainFormatter(logging.Formatter):
    def format(self, record):
        message = plain_text(super().format(record))
        if record.levelno >= logging.WARNING:
            message = f"{record.levelname}: {message}"
        return message


def configure_plain_logging():
    """Keep the library's log levels and handlers, using plain-text output."""
    from ultralytics.utils import LOGGER

    for handler in LOGGER.handlers:
        handler.setFormatter(PlainFormatter("%(message)s"))


In [ ]:
%%writefile /kaggle/working/dcrcf_v4_code/sitecustomize.py
"""Apply the notebook console format in distributed training workers."""

import os

if "LOCAL_RANK" in os.environ:
    from plain_logging import configure_plain_logging

    configure_plain_logging()


In [ ]:
%%writefile /kaggle/working/dcrcf_v4_code/warm_data.py
"""Locate stock GDUT checkpoints and the existing train-only hardaug images."""

import csv
import json
from pathlib import Path

import yaml

from gdut_data import NAMES, sha256


def dataset_yaml(prepared, output, augmented):
    prepared, output = Path(prepared), Path(output)
    train = [str(prepared / "base/images/train")]
    if augmented:
        manifest = json.loads((prepared / "augmentation_manifest.json").read_text())
        if (
            manifest.get("image_size") != 640
            or manifest.get("copies_per_train_image") != 1
            or not manifest.get("hue_saturation_transform_removed")
            or manifest.get("validation_test_augmented")
        ):
            raise ValueError("Use the GDUT color-safe 640 train-only hardaug output.")
        base = sorted((prepared / "base/images/train").glob("*.jpg"))
        hardaug = prepared / "hardaug/images/train"
        expected = {image.stem + "_hardaug.jpg" for image in base}
        if {p.name for p in hardaug.glob("*.jpg")} != expected:
            raise ValueError("Hardaug images do not match the base training images.")
        for image in hardaug.glob("*.jpg"):
            if not (
                prepared / "hardaug/labels/train" / (image.stem + ".txt")
            ).is_file():
                raise FileNotFoundError(f"Missing hardaug label: {image.stem}")
        train.append(str(hardaug))
    output.mkdir(parents=True, exist_ok=True)
    path = output / ("gdut_aug.yaml" if augmented else "gdut_base.yaml")
    path.write_text(
        yaml.safe_dump(
            dict(
                path=str(prepared),
                names=dict(enumerate(NAMES)),
                train=train,
                val=str(prepared / "base/images/val"),
                test=str(prepared / "base/images/test"),
            )
        ),
        encoding="utf-8",
    )
    return str(path)


def find_stage1(input_root, family, dataset_hash, explicit="", allow_smoke=False):
    found = []
    for path in Path(explicit or input_root).rglob("run_manifest.json"):
        value = json.loads(path.read_text())
        if (value.get("dataset"), value.get("stage"), value.get("model")) == (
            "gdut",
            "S1",
            family,
        ):
            found.append((path, value))
    if len(found) != 1:
        raise ValueError(
            f"Mount one stock GDUT Stage1 {family} output; found {len(found)}."
        )
    path, value = found[0]
    if (
        value.get("dataset_fingerprint") != dataset_hash
        or value.get("names") != NAMES
        or value.get("imgsz") != 640
    ):
        raise ValueError("Stage1 data version, classes or image size differ.")
    if value.get("smoke") and not allow_smoke:
        raise ValueError("Use a full Stage1 checkpoint.")
    expected = dict(
        optimizer="AdamW",
        lr0=0.0003,
        momentum=0.9,
        warmup_bias_lr=0.0,
        epochs=100,
        batch=16,
        imgsz=640,
        seed=3407,
    )
    if not allow_smoke:
        for key, configured in expected.items():
            if value.get("training_args", {}).get(key) != configured:
                raise ValueError(f"Stage1 does not use the CLEAN source recipe: {key}.")
    checkpoint = (path.parent / value["checkpoint"]).resolve()
    if not checkpoint.is_relative_to(path.parent.resolve()):
        raise ValueError("Checkpoint must remain inside its run directory.")
    if not checkpoint.is_file() or sha256(checkpoint) != value["checkpoint_sha256"]:
        raise ValueError("Missing or modified Stage1 checkpoint.")
    results = path.parent / "results.csv"
    if results.is_file():
        with results.open() as stream:
            value["observed_epochs_completed"] = sum(1 for _ in csv.DictReader(stream))
    return checkpoint, value


In [ ]:
%%writefile /kaggle/working/dcrcf_v4_code/warm_environment.py
"""Runtime metadata and reproducible configuration fingerprints."""

import hashlib
import importlib
import json
import os
import platform
import torch

PIN = "8.4.115"


def fingerprint(value):
    encoded = json.dumps(value, sort_keys=True, separators=(",", ":")).encode()
    return hashlib.sha256(encoded).hexdigest()


def runtime_snapshot(device, expected_gpu="Tesla T4", smoke=False):
    if device == "cpu":
        if not smoke:
            raise ValueError("Use a CUDA GPU for the full study.")
        gpus = []
    else:
        if torch.version.cuda is None:
            raise RuntimeError(
                "This session has CPU-only PyTorch. In Kaggle select Settings > "
                "Accelerator > GPU T4 x2, restart the session, then Run All."
            )
        if not torch.cuda.is_available():
            raise RuntimeError(
                "CUDA is not available in this session. Enable the Kaggle GPU "
                "accelerator and restart the session before training."
            )
        indices = [int(value) for value in str(device).split(",")]
        if len(indices) != len(set(indices)):
            raise ValueError("GPU indices must be unique.")
        count = torch.cuda.device_count()
        if any(index < 0 or index >= count for index in indices):
            raise RuntimeError(
                f"DEVICE={device!r} requests unavailable indices; this session has only "
                f"{count} GPU(s). Select GPU T4 x2 or use DEVICE='0' consistently."
            )
        gpus = []
        for index in indices:
            prop = torch.cuda.get_device_properties(index)
            if expected_gpu and expected_gpu not in prop.name:
                raise ValueError(f"Expected {expected_gpu}; found {prop.name}.")
            gpus.append(
                dict(
                    name=prop.name,
                    memory_bytes=prop.total_memory,
                    capability=[prop.major, prop.minor],
                )
            )
    versions = {}
    for name in (
        "torch",
        "torchvision",
        "ultralytics",
        "albumentations",
        "numpy",
        "scipy",
        "cv2",
        "PIL",
    ):
        try:
            versions[name] = importlib.import_module(name).__version__
        except (ImportError, AttributeError):
            if not smoke:
                raise RuntimeError(f"Install the pinned requirements; missing {name}.")
            versions[name] = "not_installed_software_check"
    if not smoke and (
        versions["ultralytics"] != PIN or versions["albumentations"] != "2.0.8"
    ):
        raise RuntimeError("Install the pinned study requirements.")
    software = dict(
        packages=versions,
        python=platform.python_version(),
        cuda=torch.version.cuda,
        cudnn=torch.backends.cudnn.version(),
        system=platform.system(),
        architecture=platform.machine(),
    )
    return dict(
        hardware=dict(gpus=gpus, cpu_architecture=platform.machine()),
        software=software,
        host=dict(cpu=platform.processor(), cpu_count=os.cpu_count()),
    )


def contract_differences(left, right, prefix=""):
    differences = []
    for key in sorted(left.keys() | right.keys()):
        path = f"{prefix}.{key}" if prefix else key
        a, b = left.get(key), right.get(key)
        if isinstance(a, dict) and isinstance(b, dict):
            differences.extend(contract_differences(a, b, path))
        elif a != b:
            differences.append(dict(field=path, left=a, right=b))
    return differences


In [ ]:
%%writefile /kaggle/working/dcrcf_v4_code/warm_launch.py
"""Run a training subprocess, retain the full log, and keep notebook output concise."""

from collections import deque
from pathlib import Path
import re
import subprocess


_PROGRESS = re.compile(r"(?:\d+%|\d+/\d+.*(?:it/s|s/it))")
_FINAL_COUNTER = re.compile(r"(?:159/159|2530/2530|316/316|1587/1587)")
_IMPORTANT = re.compile(
    r"(?:Traceback|Error|Exception|WARNING|EarlyStopping|Results saved|"
    r"Baseline preflight|Baseline fine-tuning complete|Transferred|Starting training|"
    r"epochs completed|checkpoint|run_manifest|validation_details|model summary|parameters|GFLOPs|optimizer|Image sizes)",
    re.IGNORECASE,
)


def _display_line(line, emitted_once):
    """Return a concise display line while the original line is always logged."""
    text = line.rstrip("\r\n")
    if not text:
        return None
    if "compute_grad_input does not have a deterministic implementation" in text:
        key = "dcnv2_nondeterminism_warning"
        if key in emitted_once:
            return None
        emitted_once.add(key)
        return "WARNING: DCNv2 backward is not bitwise deterministic on this CUDA stack."
    if _PROGRESS.search(text) and not _FINAL_COUNTER.search(text):
        return None
    if _FINAL_COUNTER.search(text) or re.match(r"^\s*all\s+\d+", text):
        return text
    if _IMPORTANT.search(text):
        return text
    return None


def run_logged(command, log_path):
    """Execute command; write every byte to disk and emit only audit-relevant lines."""
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    arguments = list(command)
    if "-u" not in arguments[1:]:
        arguments.insert(1, "-u")

    tail = deque(maxlen=50)
    emitted_once = set()
    with log_path.open("w", encoding="utf-8") as log:
        with subprocess.Popen(
            arguments,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8",
            errors="replace",
            bufsize=1,
        ) as process:
            try:
                for line in process.stdout:
                    log.write(line)
                    log.flush()
                    tail.append(line)
                    display = _display_line(line, emitted_once)
                    if display is not None:
                        print(display, flush=True)
                returncode = process.wait()
            except BaseException:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
                raise

    if returncode:
        raise RuntimeError(
            f"Training subprocess exited with code {returncode}. Full log: {log_path}\n"
            + "".join(tail)
        )
    print(f"Full training log: {log_path}", flush=True)
    return log_path


In [ ]:
%%writefile /kaggle/working/dcrcf_v4_code/warm_metrics.py
"""Export AP, precision, recall and class-wise F1 with explicit confidence rules."""

import copy
import time

import numpy as np
import torch

from custom_ablation_modules import RepConv
from gdut_data import NAMES


def f1_macro(precision, recall):
    precision, recall = np.asarray(precision), np.asarray(recall)
    values = np.divide(
        2 * precision * recall,
        precision + recall,
        out=np.zeros_like(precision, dtype=float),
        where=precision + recall > 0,
    )
    return float(values.mean()) if values.size else 0.0


def export_metrics(result, confidence=0.25):
    box = result.box
    summary = dict(
        precision=float(box.mp),
        recall=float(box.mr),
        mAP50=float(box.map50),
        mAP50_95=float(box.map),
        F1_native_macro=float(np.asarray(box.f1).mean()) if len(box.f1) else 0.0,
    )
    per_class, p_fixed, r_fixed = [], [], []
    for index, class_id in enumerate(box.ap_class_index):
        p = float(np.interp(confidence, box.px, box.p_curve[index]))
        r = float(np.interp(confidence, box.px, box.r_curve[index]))
        p_fixed.append(p)
        r_fixed.append(r)
        per_class.append(
            dict(
                class_id=int(class_id),
                name=NAMES[class_id],
                precision=float(box.p[index]),
                recall=float(box.r[index]),
                F1_native=float(box.f1[index]),
                AP50=float(box.all_ap[index, 0]),
                AP50_95=float(box.all_ap[index].mean()),
                precision_conf025=p,
                recall_conf025=r,
                F1_conf025=f1_macro([p], [r]),
            )
        )
    present = {row["class_id"] for row in per_class}
    for class_id, name in enumerate(NAMES):
        if class_id not in present:
            per_class.append(
                dict(
                    class_id=class_id,
                    name=name,
                    precision=0.0,
                    recall=0.0,
                    F1_native=0.0,
                    AP50=0.0,
                    AP50_95=0.0,
                    precision_conf025=0.0,
                    recall_conf025=0.0,
                    F1_conf025=0.0,
                )
            )
            p_fixed.append(0.0)
            r_fixed.append(0.0)
    per_class.sort(key=lambda row: row["class_id"])
    summary.update(
        precision_conf025_macro=float(np.mean(p_fixed)),
        recall_conf025_macro=float(np.mean(r_fixed)),
        F1_conf025_macro=f1_macro(p_fixed, r_fixed),
    )
    definitions = dict(
        precision_recall="Ultralytics class means at the native operating point",
        F1_native_macro="Mean class F1 at the native mean-F1 operating point; IoU 0.5",
        F1_conf025_macro="Mean class F1 interpolated from PR/confidence curves at confidence 0.25; IoU 0.5",
        AP="Ultralytics 101-point AP integration; AP50 and mean AP at IoU 0.50:0.05:0.95",
    )
    return summary, per_class, definitions


def benchmark_forward(model, device, smoke=False):
    target = torch.device("cpu" if device == "cpu" else "cuda:" + str(device))
    measured = copy.deepcopy(model).to(target).float().eval()
    converted = 0
    for layer in list(measured.modules()):
        if isinstance(layer, RepConv):
            layer.switch_to_deploy()
            converted += 1
    measured.fuse(verbose=False)
    tensor = torch.zeros(1, 3, 640, 640, device=target)
    iterations, warmup = (2, 1) if smoke else (30, 10)

    def sync():
        if target.type == "cuda":
            torch.cuda.synchronize(target)

    times = []
    with torch.inference_mode():
        for _ in range(warmup):
            measured(tensor)
        sync()
        for _ in range(iterations):
            sync()
            start = time.perf_counter()
            measured(tensor)
            sync()
            times.append(1000 * (time.perf_counter() - start))
    return dict(
        forward_ms_median=float(np.median(times)),
        forward_ms_mean=float(np.mean(times)),
        batch=1,
        imgsz=640,
        precision="float32",
        includes_preprocessing=False,
        includes_nms=False,
        warmup=warmup,
        iterations=iterations,
        device=str(target),
        hardware=torch.cuda.get_device_name(target) if target.type == "cuda" else "CPU",
        repconv_deploy_blocks=converted,
        deployment_parameters=sum(p.numel() for p in measured.parameters()),
    )


In [ ]:
%%writefile /kaggle/working/dcrcf_v4_code/warm_model.py
"""Load stock Stage1 weights before inserting the baseline modules."""

import json
import os

import torch
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import RANK

from baseline_model import PaperBaselineTrainer, apply_baseline, module_inventory
from warm_protocol import MODELS, REVISION


def validate_stock_model(model, identifier):
    if hasattr(model, "research_variant") or hasattr(model, "baseline_id"):
        raise ValueError(
            "Use a stock Stage1 checkpoint, not an A6 or baseline checkpoint."
        )
    expected = MODELS[identifier]
    classes = {type(layer).__name__ for layer in model.model}
    family = (
        "yolo11" if "C2PSA" in classes else "yolov8" if "C2f" in classes else "unknown"
    )
    expected_family = "yolo11" if expected.startswith("yolo11") else "yolov8"
    stem = model.model[0].conv.out_channels
    if family != expected_family or stem != (16 if expected == "yolo11n" else 32):
        raise ValueError(f"Expected the {expected} stock Stage1 scale.")
    if model.model[-1].nc != 5:
        raise ValueError("Stage1 must use the five GDUT classes.")


def transfer_baseline(model, identifier):
    validate_stock_model(model, identifier)
    previous_parameters = {
        id(value): (name, value) for name, value in model.named_parameters()
    }
    previous_buffers = {
        id(value): (name, value) for name, value in model.named_buffers()
    }
    pooling = model.model[9] if identifier.startswith("yolov8") else None
    apply_baseline(model, identifier)
    explicit = {}
    if pooling is not None:
        target = model.model[9].branches[0]
        with torch.no_grad():
            target[0].weight.copy_(pooling.cv1.conv.weight)
            target[1].load_state_dict(pooling.cv1.bn.state_dict())
        target[1].eps, target[1].momentum = pooling.cv1.bn.eps, pooling.cv1.bn.momentum
        explicit["model.9.branches.0.0.weight"] = "model.9.cv1.conv.weight"
        for key in pooling.cv1.bn.state_dict():
            explicit["model.9.branches.0.1." + key] = "model.9.cv1.bn." + key
    mapping, initialized, transferred_elements = [], [], 0
    for name, parameter in model.named_parameters():
        retained = previous_parameters.get(id(parameter))
        source_name = retained[0] if retained is not None else explicit.get(name)
        if source_name is not None:
            mapping.append(
                dict(
                    target=name,
                    source=source_name,
                    shape=list(parameter.shape),
                    kind="parameter",
                )
            )
            transferred_elements += parameter.numel()
        else:
            initialized.append(dict(name=name, shape=list(parameter.shape)))
    for name, buffer in model.named_buffers():
        retained = previous_buffers.get(id(buffer))
        source_name = retained[0] if retained is not None else explicit.get(name)
        if source_name is not None:
            mapping.append(
                dict(
                    target=name,
                    source=source_name,
                    shape=list(buffer.shape),
                    kind="buffer",
                )
            )
    audit = dict(
        source_model=MODELS[identifier],
        transferred=mapping,
        newly_initialized_parameters=initialized,
        transferred_parameter_elements=transferred_elements,
        total_parameter_elements=sum(p.numel() for p in model.parameters()),
        new_modules_initialization="native baseline initializers",
    )
    model.transfer_audit = audit
    model.warm_revision = REVISION
    return model, audit


class WarmTrainer(PaperBaselineTrainer):
    baseline_id = "yolo11_dcrcf"

    def get_model(self, cfg=None, weights=None, verbose=True):
        if weights is None:
            raise ValueError("A stock GDUT Stage1 checkpoint is required.")
        validate_stock_model(weights, self.baseline_id)
        model = self.set_model_names_for_load(
            DetectionModel(
                cfg, nc=self.data["nc"], ch=self.data["channels"], verbose=verbose
            )
        )
        model.load(weights)
        model, audit = transfer_baseline(model, self.baseline_id)
        source_state, target_state = weights.state_dict(), model.state_dict()
        for item in audit["transferred"]:
            target = target_state[item["target"]]
            expected = source_state[item["source"]].to(
                device=target.device, dtype=target.dtype
            )
            if not torch.equal(target, expected):
                raise ValueError(f"Transferred tensor differs: {item['target']}")
        report = dict(
            baseline_id=self.baseline_id,
            revision=REVISION,
            rank=os.environ.get("RANK", "-1"),
            modules=module_inventory(model),
            transfer=audit,
        )
        self.save_dir.mkdir(parents=True, exist_ok=True)
        (self.save_dir / f"architecture_rank_{report['rank']}.json").write_text(
            json.dumps(report, indent=2), encoding="utf-8"
        )
        return model

    def final_eval(self):
        super().final_eval()
        if RANK in {-1, 0}:
            path = self.save_dir / "training_complete.json"
            value = json.loads(path.read_text())
            value["revision"] = REVISION
            path.write_text(json.dumps(value, indent=2), encoding="utf-8")


class WarmDCRCFTrainer(WarmTrainer):
    baseline_id = "yolo11_dcrcf"


class WarmDWRTrainer(WarmTrainer):
    baseline_id = "yolov8_dwr_aspp_nwd"


TRAINERS = {"yolo11_dcrcf": WarmDCRCFTrainer, "yolov8_dwr_aspp_nwd": WarmDWRTrainer}


In [ ]:
%%writefile /kaggle/working/dcrcf_v4_code/warm_protocol.py
"""Stage1 and fine-tuning settings shared with the CLEAN A6 recipe."""

REVISION = "gdut_stage1_baselines_v1"
MODELS = {"yolo11_dcrcf": "yolo11n", "yolov8_dwr_aspp_nwd": "yolov8s"}
COMMON = {
    "epochs": 100,
    "batch": 16,
    "imgsz": 640,
    "seed": 3407,
    "optimizer": "AdamW",
    "lr0": 0.0001,
    "lrf": 0.01,
    "momentum": 0.9,
    "weight_decay": 0.0005,
    "pretrained": True,
    "resume": False,
    "patience": 50,
    "deterministic": True,
    "workers": 4,
    "amp": True,
    "compile": False,
    "cos_lr": False,
    "warmup_epochs": 3.0,
    "warmup_momentum": 0.8,
    "warmup_bias_lr": 0.0,
    "nbs": 64,
    "box": 7.5,
    "cls": 0.5,
    "dfl": 1.5,
    "hsv_h": 0.0,
    "hsv_s": 0.0,
    "hsv_v": 0.4,
    "bgr": 0.0,
    "degrees": 0.0,
    "translate": 0.1,
    "scale": 0.5,
    "shear": 0.0,
    "perspective": 0.0,
    "flipud": 0.0,
    "fliplr": 0.5,
    "mosaic": 1.0,
    "close_mosaic": 10,
    "mixup": 0.0,
    "cutmix": 0.0,
    "copy_paste": 0.0,
    "save": True,
    "save_period": -1,
    "plots": True,
    "cache": False,
    "val": True,
    "rect": False,
    "fraction": 1.0,
    "multi_scale": False,
}
EVALUATION = dict(
    imgsz=640,
    batch=16,
    conf=0.001,
    iou=0.7,
    max_det=300,
    workers=0,
    half=False,
    quantize=None,
    augment=False,
    plots=True,
)


def training_settings(device="0,1", stage1=False, smoke=False):
    values = dict(COMMON, device=device)
    if stage1:
        values["lr0"] = 0.0003
    if smoke:
        values.update(epochs=1, batch=2, imgsz=64, workers=0, plots=False, amp=False)
    return values


In [ ]:
%%writefile /kaggle/working/dcrcf_v4_code/v4_data.py
"""Validate the mounted GDUT split, labels and offline augmentation."""

from collections import Counter, defaultdict
from hashlib import sha256 as digest
from pathlib import Path

import numpy as np
from PIL import Image
import yaml

from gdut_data import NAMES, prepared_dataset, sha256
from warm_data import dataset_yaml, find_stage1

PREPARED = Path("/kaggle/input/notebooks/nvthanh2004/gdut-b-s-1-5-class-dataset-preparation/prepared_gdut")
STAGE1 = Path("/kaggle/input/notebooks/nvthanh2004/gdut-stock-yolo11n-stage1/stage1_runs/gdut_S1_yolo11n")
DATA_HASH = "026a2afb617254b9dd2b9461651bce6f74295d18f29d52a790bf755d32944f65"
AUG_HASH = "95318f647ea1493b9d83779d150abf94ad4a6c32c54f3e3505a86f078d837342"
STAGE1_HASH = "ea174f78b6b768a2f436c4248a7b95e3001ca3ff47fe2d850a78c22f3e98a88c"
V8_STAGE1 = Path("/kaggle/input/notebooks/nvthanh2004/gdut-b-s-1-re-parameterized-yolo-architecture1/runs/gdut_S1_yolov8s")
V8_STAGE1_HASH = "af430179fd7c541f666728b9a2539d03ce1909210d21c4ecf8aba74e73e948d2"
STAGE1_CONTRACTS = {"yolo11n": (STAGE1, STAGE1_HASH), "yolov8s": (V8_STAGE1, V8_STAGE1_HASH)}
SPLITS = {"train": 1265, "val": 316, "test": 1587}


def labels(path):
    rows = []
    for line in path.read_text().splitlines():
        cls, x, y, w, h = map(float, line.split())
        if not (np.isfinite([cls, x, y, w, h]).all() and cls in range(5)
                and w > 0 and h > 0 and min(x-w/2, y-h/2) >= -2e-6
                and max(x+w/2, y+h/2) <= 1+2e-6):
            raise ValueError(f"Invalid YOLO label: {path}")
        rows.append((int(cls), x, y, w, h))
    return rows


def check_split(root, records, section, split, file_hashes, pixel_splits):
    path = root / section
    suffix = "_hardaug" if section == "hardaug" else ""
    images = path / "images" / split
    targets = path / "labels" / split
    ids = {r["id"] + suffix for r in records}
    for directory, extension in ((images, ".jpg"), (targets, ".txt")):
        actual = {p.name for p in directory.iterdir() if p.is_file() and p.suffix.lower() != ".cache"}
        if actual != {stem + extension for stem in ids}:
            raise ValueError(f"File coverage mismatch in {directory}")
    count = Counter()
    empty = []
    dropped = Counter()
    for record in sorted(records, key=lambda r: r["id"]):
        stem = record["id"] + suffix
        image, target = images / (stem + ".jpg"), targets / (stem + ".txt")
        rows = labels(target)
        current = Counter(row[0] for row in rows)
        source = Counter(int(row[0]) for row in record["boxes"])
        count.update(current)
        if not rows:
            empty.append(stem)
        with Image.open(image) as opened:
            rgb = opened.convert("RGB")
            pixel_hash = digest(str(rgb.size).encode() + rgb.tobytes()).hexdigest()
        pixel_splits[pixel_hash].add(split)
        if section == "base":
            if pixel_hash != record["pixel_hash"] or current != source or not np.allclose(
                rows, record["boxes"], atol=1e-9, rtol=0
            ):
                raise ValueError(f"Base pixels or labels differ from manifest: {stem}")
        elif rgb.size != (640, 640) or current - source:
            raise ValueError(f"Hardaug dimensions or classes differ: {stem}")
        dropped.update(source - current)
        for file in (image, target):
            file_hashes[file.relative_to(root).as_posix()] = sha256(file)
    return {"images": len(ids), "boxes": {name: count[i] for i, name in enumerate(NAMES)},
            "box_count_reduction": {name: dropped[i] for i, name in enumerate(NAMES)},
            "empty_labels": empty}


def validate_inputs(work, prepared=PREPARED, stage1=None, include_test=False, family="yolo11n"):
    root, manifest, fingerprint = prepared_dataset(str(prepared))
    if fingerprint != DATA_HASH or sha256(root / "augmentation_manifest.json") != AUG_HASH:
        raise ValueError("GDUT manifest differs from V3.")
    mount, expected_checkpoint = STAGE1_CONTRACTS[family]
    weight, source = find_stage1(stage1 or mount, family, DATA_HASH)
    if source["checkpoint_sha256"] != expected_checkpoint:
        raise ValueError(f"Stage1 checkpoint differs from the audited {family} source.")
    groups = defaultdict(list)
    seen = {field: {} for field in ("id", "group", "pixel_hash")}
    for row in manifest["records"]:
        split = row["split"]
        groups[split].append(row)
        if (row["official"] == "test") != (split == "test"):
            raise ValueError("Official split assignment changed.")
        for field, index in seen.items():
            previous = index.setdefault(row[field], split)
            if previous != split:
                raise ValueError(f"Cross-split overlap in {field}")
    if {key: len(rows) for key, rows in groups.items()} != SPLITS:
        raise ValueError("Unexpected GDUT split counts.")
    if len(seen["id"]) != sum(SPLITS.values()):
        raise ValueError("Duplicate image IDs.")
    config = Path(dataset_yaml(root, work, True))
    file_hashes, pixel_splits = {}, defaultdict(set)
    report = {}
    for split in ("train", "val"):
        report[split] = check_split(root, groups[split], "base", split, file_hashes, pixel_splits)
    report["hardaug"] = check_split(root, groups["train"], "hardaug", "train", file_hashes, pixel_splits)
    if include_test:
        test_hashes = {}
        report["test"] = check_split(root, groups["test"], "base", "test", test_hashes, pixel_splits)
        report["test_sha256"] = test_hashes
    else:
        for row in groups["test"]:
            pixel_splits[row["pixel_hash"]].add("test")
    if any(len(splits) != 1 for splits in pixel_splits.values()):
        raise ValueError("Cross-split pixel overlap in effective dataset.")
    report["train_val_sha256"] = file_hashes
    report["manifest_sha256"] = fingerprint
    report["augmentation_manifest_sha256"] = AUG_HASH
    report["stage1_checkpoint_sha256"] = expected_checkpoint
    report["stage1_family"] = family
    report["expected_splits"] = SPLITS
    report["effective_train_samples"] = report["train"]["images"] + report["hardaug"]["images"]
    if not include_test:
        data = yaml.safe_load(config.read_text())
        data.pop("test")
        config.write_text(yaml.safe_dump(data, sort_keys=False))
    return config, weight, report


def make_smoke_config(config, work, pairs=32):
    """Use a reproducible base/hardaug paired subset; keep full validation."""
    data = yaml.safe_load(config.read_text())
    base, hardaug = map(Path, data["train"])
    candidates = sorted(base.glob("*.jpg"),
                        key=lambda p: (digest(f"smoke-v4:{p.stem}".encode()).digest(), p.stem))
    if len(candidates) < pairs:
        raise ValueError(f"Smoke requires at least {pairs} base/hardaug pairs.")
    def classes(path):
        return {row[0] for row in labels(path.parents[2] / "labels" / "train" / (path.stem + ".txt"))}

    # Start with witnesses for all classes in both views; then fill by hash.
    selected = {}
    for directory in (base, hardaug):
        missing = set(range(len(NAMES)))
        for path in candidates:
            sample = path if directory == base else directory / f"{path.stem}_hardaug.jpg"
            present = classes(sample)
            if present & missing:
                selected[path] = None
                missing -= present
            if not missing:
                break
        if missing:
            raise ValueError(f"Smoke has no class witness in {directory}: {sorted(missing)}")
    if len(selected) > pairs:
        raise ValueError("Smoke pair budget is too small for class coverage.")
    for path in candidates:
        if len(selected) == pairs:
            break
        selected[path] = None
    images = [image for path in selected
              for image in (path, hardaug / f"{path.stem}_hardaug.jpg")]
    if len(set(images)) != 2 * pairs or not all(path.is_file() for path in images):
        raise ValueError("Smoke dataset is missing a base/hardaug pair.")
    listing = work / "smoke_train_images.txt"
    listing.write_text("".join(f"{path}\n" for path in images))
    data["train"] = str(listing)
    output = work / "gdut_smoke.yaml"
    output.write_text(yaml.safe_dump(data, sort_keys=False))
    return output


In [ ]:
%%writefile /kaggle/working/dcrcf_v4_code/v4_evaluate.py
"""Verify the full paired matrix before evaluating its frozen GDUT test split."""

import argparse
from collections import defaultdict
import csv
import json
import math
from pathlib import Path
from statistics import mean, stdev

from scipy.stats import t
import torch
from ultralytics import YOLO

from gdut_data import sha256
from gdut_data import NAMES
from v4_data import PREPARED, validate_inputs
from v4_model import ARMS, V8_ARMS, assert_graph
from v4_run import SEEDS, code_hashes, dump, evaluate
from warm_environment import fingerprint, runtime_snapshot
from warm_metrics import benchmark_forward
from warm_protocol import COMMON, EVALUATION

THRESHOLD = 0.25


def collect(input_root, arms=tuple(ARMS)):
    records = {}
    smoke_receipts = set()
    for arm in arms:
        slug = ("gdut-yolov8-v4-" if arm in V8_ARMS else "gdut-dcrcf-v4-") + arm.lower()
        base = input_root / "notebooks" / "tundng111" / slug / f"v4-{arm.lower()}"
        marker = json.loads((base / "ADD_INPUT_READY.json").read_text())
        if (marker.get("status"), marker.get("arm"), marker.get("seeds")) != ("ADD_INPUT_READY", arm, list(SEEDS)):
            raise ValueError(f"Wrong Add Input marker: {base}")
        for relative, expected in marker["artifacts"].items():
            path = (base / relative).resolve()
            if not path.is_relative_to(base.resolve()) or sha256(path) != expected:
                raise ValueError(f"Artifact hash mismatch: {relative}")
        if f"v4-{arm.lower()}.log" not in marker["artifacts"]:
            raise ValueError(f"Missing full subprocess log: {base}")
        if "cuda_smoke_verified.json" not in marker["artifacts"]:
            raise ValueError(f"Missing CUDA smoke verification: {base}")
        smoke_receipts.add(json.loads((base / "cuda_smoke_verified.json").read_text())["receipt_sha256"])
        for seed in SEEDS:
            folder = base / f"{arm}-{seed}"
            manifest_path = folder / "run_manifest.json"
            record = json.loads(manifest_path.read_text())
            if (record["revision"], record["track"], record["arm"], record["seed"], record["smoke"]) != (
                "dcrcf_v4", "gdut", arm, seed, False
            ):
                raise ValueError(f"Unexpected run identity: {folder}")
            expected_args = dict(COMMON, seed=seed, device="0,1")
            if any(record["args"].get(k) != value for k, value in expected_args.items()):
                raise ValueError(f"Run recipe differs from the locked V4 protocol: {folder}")
            if record["execution"] != {"amp_active": True, "world_size": 2}:
                raise ValueError(f"Unexpected execution settings: {folder}")
            if record["checkpoint_selection"] != "max_native_validation_mAP50_95_latest_exact_tie":
                raise ValueError(f"Unexpected checkpoint selection rule: {folder}")
            required_files = ["run_manifest.json", "args.yaml", "results.csv",
                              "training_complete.json", "validation/predictions.json",
                              "validation/metrics.json", "validation/module_probe.json",
                              "weights/best.pt", "weights/last.pt"]
            if any(f"{folder.name}/{name}" not in marker["artifacts"] for name in required_files):
                raise ValueError(f"Add Input marker does not cover every required artifact: {folder}")
            path = folder / "weights/best.pt"
            expected = record["checkpoint_sha256"]["weights/best.pt"]
            if sha256(path) != expected or record["code"] != code_hashes():
                raise ValueError(f"Checkpoint or source drift: {folder}")
            records[arm, seed] = (path, record)
    if len(smoke_receipts) != 1:
        raise ValueError("Training notebooks used different smoke receipts.")
    reference = None
    for path, record in records.values():
        contract = dict(inputs=record["inputs"], code=record["code"],
                        software=record["runtime"]["software"], hardware=record["runtime"]["hardware"],
                        args={k: v for k, v in record["args"].items() if k not in {
                            "seed", "project", "name", "save_dir", "data"
                        }})
        if reference is not None and contract != reference:
            raise ValueError(f"Matched protocol drift: {path}")
        reference = contract
    return records


def paired_summary(results, arms=tuple(ARMS)):
    def interval(values):
        center = mean(values)
        sd = stdev(values)
        margin = t.ppf(.975, len(values) - 1) * sd / math.sqrt(len(values))
        return dict(mean=center, sd=sd, ci95=[center-margin, center+margin], values=values)

    effects = {
        "full_minus_stock": {"C3": 1, "C0": -1},
        "dcn_alone": {"C1": 1, "C0": -1},
        "rcf_alone": {"C2": 1, "C0": -1},
        "interaction": {"C3": 1, "C2": -1, "C1": -1, "C0": 1},
    }
    if tuple(arms) == V8_ARMS:
        effects = {"full_minus_stock": {"V8F": 1, "V8S": -1}}
    return {
        "primary": "mAP50_95",
        "uncertainty": "paired t interval across five training seeds, conditional on the fixed split and Stage1 checkpoint",
        "metrics": {
            metric: {
                "arms": {arm: interval([results[arm, seed][metric] for seed in SEEDS]) for arm in arms},
                "paired": {
                    name: interval([sum(sign * results[arm, seed][metric] for arm, sign in coeff.items()) for seed in SEEDS])
                    for name, coeff in effects.items()
                },
            }
            for metric in next(iter(results.values()))
        },
    }


def iou(a, b):
    x = max(0., min(a[0]+a[2], b[0]+b[2])-max(a[0], b[0]))
    y = max(0., min(a[1]+a[3], b[1]+b[3])-max(a[1], b[1]))
    intersection = x*y
    return intersection / max(a[2]*a[3] + b[2]*b[3] - intersection, 1e-12)


def diagnostics(predictions, truth):
    grouped = defaultdict(list)
    for detection in predictions:
        image_id = str(detection["image_id"])
        if image_id not in truth:
            raise ValueError(f"Prediction references unknown image {image_id}.")
        # The pinned validator exports non-COCO category IDs starting at one.
        if detection["category_id"] not in range(1, 6):
            raise ValueError(f"Unexpected category ID: {detection['category_id']}")
        if detection["score"] >= THRESHOLD:
            grouped[image_id].append(detection)
    counts = defaultdict(lambda: dict(gt=0, tp=0, fn=0, fp=0))
    confusion = [[0] * 5 for _ in range(5)]
    duplicate_fp = 0
    background_fp = 0
    for image_id, row in truth.items():
        width, height = row["width"], row["height"]
        gain = 640 / max(width, height)
        objects = []
        for category, cx, cy, w, h in row["boxes"]:
            box = [(cx-w/2)*width, (cy-h/2)*height, w*width, h*height]
            is_small = box[2]*box[3]*gain**2 < 32**2
            buckets = ("all", "small" if is_small else "other_size", f"class_{int(category)}")
            objects.append((int(category), box, buckets))
            for bucket in buckets:
                counts[bucket]["gt"] += 1
        matched = set()
        unmatched = []
        for pred in sorted(grouped[image_id], key=lambda p: -p["score"]):
            candidates = [(iou(box, pred["bbox"]), index)
                          for index, (category, box, _) in enumerate(objects)
                          if index not in matched and category + 1 == pred["category_id"]]
            overlap, index = max(candidates, default=(0, -1))
            if overlap >= 0.5:
                matched.add(index)
                for bucket in objects[index][2]:
                    counts[bucket]["tp"] += 1
            else:
                counts["all"]["fp"] += 1
                unmatched.append(pred)
        # Diagnostic only. Match leftover predictions to leftover GT after
        # same-class TPs, so wrong-label overlaps do not count as detection TPs.
        for pred in unmatched:
            overlaps = [(iou(box, pred["bbox"]), index)
                        for index, (_, box, _) in enumerate(objects)]
            overlap, index = max((item for item in overlaps if item[1] not in matched),
                                 default=(0, -1))
            if overlap >= 0.5:
                confusion[objects[index][0]][pred["category_id"] - 1] += 1
                matched.add(index)
            elif any(value >= 0.5 for value, _ in overlaps):
                duplicate_fp += 1
            else:
                background_fp += 1
        for bucket in counts.values():
            bucket["fn"] = bucket["gt"] - bucket["tp"]
    for name, bucket in counts.items():
        if name != "all":
            bucket.pop("fp")
        bucket["recall"] = bucket["tp"] / bucket["gt"] if bucket["gt"] else None
    return dict(confidence=THRESHOLD, iou=0.5, size_rule="box area after scaling to 640, small < 1024 px²",
                counts=counts, class_confusion_gt_by_pred=confusion,
                fp_overlapping_already_matched_gt=duplicate_fp, fp_without_gt_iou50=background_fp,
                unavailable=["night", "occlusion", "distractor labels"])


def main(work, input_root, family="yolo11n"):
    arms = tuple(ARMS) if family == "yolo11n" else V8_ARMS
    records = collect(input_root, arms)
    runtime = runtime_snapshot("0", "Tesla T4")
    if any(row["runtime"]["software"] != runtime["software"] for _, row in records.values()):
        raise ValueError("Evaluator software differs from training.")
    work.mkdir(parents=True, exist_ok=False)
    dump(work / "test_lock.json", dict(
        policy=f"all {len(arms) * len(SEEDS)} predeclared arms/seeds; model selected on validation fitness",
        settings=EVALUATION, diagnostic_confidence=THRESHOLD, evaluator_code=code_hashes(),
        checkpoints={f"{a}-{s}": row["checkpoint_sha256"]["weights/best.pt"]
                     for (a, s), (_, row) in records.items()},
        historical_exposure="This GDUT holdout has appeared in earlier project results.",
    ))
    config, _, audit = validate_inputs(work, include_test=True, family=family)
    verified_inputs = {key: audit[key] for key in (
        "manifest_sha256", "augmentation_manifest_sha256", "stage1_checkpoint_sha256"
    )}
    verified_inputs["train_val_contents"] = fingerprint(audit["train_val_sha256"])
    if any(row["inputs"] != verified_inputs for _, row in records.values()):
        raise ValueError("Mounted dataset content differs from training.")
    dump(work / "data_audit.json", audit)
    truth = {str(int(row["id"])) if row["id"].isnumeric() else row["id"]: row
             for row in json.loads((PREPARED / "dataset_manifest.json").read_text())["records"]
             if row["split"] == "test"}
    results = {}
    for (arm, seed), (path, _) in records.items():
        model = YOLO(str(path))
        assert_graph(model.model, arm)
        if model.model.v4_arm != arm:
            raise ValueError(f"Checkpoint identifies a different arm: {path}")
        target = work / f"{arm}-{seed}"
        summary = evaluate(model, config, "test", target, "0")
        details = json.loads((target / "metrics.json").read_text())
        none = next(row for row in details["per_class"] if row["name"] == "none")
        summary["none_AP50_95"] = none["AP50_95"]
        predictions = json.loads((target / "predictions.json").read_text())
        buckets = diagnostics(predictions, truth)
        dump(target / "error_buckets.json", buckets)
        small = buckets["counts"].get("small", {}).get("recall")
        if small is None:
            raise ValueError("The locked test split has no small-object support.")
        summary["small_recall_conf025"] = small
        summary["none_recall_conf025"] = buckets["counts"][f"class_{NAMES.index('none')}"]["recall"]
        for row in details["per_class"]:
            key = row["name"]
            summary[f"{key}_AP50"] = row["AP50"]
            summary[f"{key}_AP50_95"] = row["AP50_95"]
            summary[f"{key}_recall_native"] = row["recall"]
        model.model.cpu()
        torch.cuda.synchronize(0)
        benchmark_baseline = torch.cuda.memory_allocated(0)
        torch.cuda.reset_peak_memory_stats(0)
        benchmark = benchmark_forward(model.model, "0")
        peak = torch.cuda.max_memory_allocated(0)
        benchmark.update(torch_cuda_baseline_allocated_bytes=benchmark_baseline,
                         torch_cuda_peak_allocated_bytes=peak,
                         torch_cuda_peak_increment_bytes=peak - benchmark_baseline)
        dump(target / "benchmark.json", benchmark)
        summary["forward_ms_median"] = benchmark["forward_ms_median"]
        summary["torch_cuda_peak_increment_bytes"] = benchmark["torch_cuda_peak_increment_bytes"]
        results[arm, seed] = summary
        del model
        torch.cuda.empty_cache()
    dump(work / "paired_summary.json", paired_summary(results, arms))
    with (work / "test_metrics.csv").open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=["arm", "seed", *next(iter(results.values()))])
        writer.writeheader()
        writer.writerows(dict(arm=arm, seed=seed, **row) for (arm, seed), row in results.items())


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--work", type=Path, required=True)
    parser.add_argument("--input-root", type=Path, default=Path("/kaggle/input"))
    parser.add_argument("--family", choices=("yolo11n", "yolov8s"), default="yolo11n")
    args = parser.parse_args()
    main(args.work, args.input_root, args.family)


In [ ]:
%%writefile /kaggle/working/dcrcf_v4_code/v4_model.py
"""Factorial YOLO11n models with explicit stock-weight provenance."""

import json
import os

import torch
from torch import nn
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import RANK

from baseline_model import CIoUNWDBboxLoss, PaperBaselineTrainer, apply_baseline, graph_layer, module_inventory
from baseline_modules import ModulatedDCNv2
from gdut_data import NAMES
from warm_model import transfer_baseline, validate_stock_model

ARMS = {"C0": (False, False), "C1": (True, False), "C2": (False, True), "C3": (True, True)}
V8_ARMS = ("V8S", "V8F")
V8_BASELINE = "yolov8_dwr_aspp_nwd"


def family_for(arm):
    if arm in ARMS:
        return "yolo11n"
    if arm in V8_ARMS:
        return "yolov8s"
    raise ValueError(f"Unknown experiment arm: {arm}")


class DCNStockLossModel(DetectionModel):
    """Native YOLO loss with a summary that never profiles deformable convolution."""

    def info(self, detailed=False, verbose=True, imgsz=640):
        # Native model_info calls THOP when verbose. Graph/MAC evidence is
        # produced by the explicit forward hooks in the preflight instead.
        return super().info(detailed=detailed, verbose=False, imgsz=imgsz)


def inventory(model):
    return {key: len(paths) for key, paths in module_inventory(model).items()}


def assert_graph(model, arm):
    if family_for(arm) == "yolov8s":
        full = arm == "V8F"
        expected = dict(DWR=3 * full, ASPP=int(full), ModulatedDCNv2=0, SBA=0)
        if len(model.model) != 23:
            raise ValueError("Expected the 23-layer YOLOv8s graph.")
        if full:
            if (any(type(model.model[i]).__name__ != "C2fWithDWR" for i in (2, 4, 6))
                    or type(model.model[9]).__name__ != "ASPP"
                    or (model.nwd_constant, model.nwd_weight) != (12.8, 0.5)):
                raise ValueError("YOLOv8 module placement or NWD configuration differs.")
        elif type(model.model[9]).__name__ != "SPPF":
            raise ValueError("Stock YOLOv8s must retain SPPF.")
    else:
        dcn, rcf = ARMS[arm]
        expected = dict(DWR=0, ASPP=0, ModulatedDCNv2=2 * dcn, SBA=4 * rcf)
    if inventory(model) != expected or model.stride.tolist() != [8, 16, 32]:
        raise ValueError(f"{arm} graph does not match its contract.")


def construct(model, arm, source=None):
    dcn, rcf = ARMS.get(arm, (False, False))
    identifier = V8_BASELINE if family_for(arm) == "yolov8s" else "yolo11_dcrcf"
    if source is not None:
        validate_stock_model(source, identifier)
        if source.names != dict(enumerate(NAMES)):
            raise ValueError("Stage1 checkpoint class order differs from GDUT.")
        model.load(source)
    original = {id(tensor): (name, tensor) for name, tensor in model.state_dict(keep_vars=True).items()}
    explicit = {}
    if arm == "V8F":
        if source is None:
            raise ValueError("GDUT YOLOv8s full requires the audited stock Stage1 source.")
        with torch.random.fork_rng(devices=[]):
            torch.random.default_generator.manual_seed((torch.initial_seed() + 8100) % 2**63)
            model, audit = transfer_baseline(model, V8_BASELINE)
        explicit = {row["target"]: row["source"] for row in audit["transferred"]}
    elif rcf:
        # V3 is the canonical RCF reconstruction. In C2 retain its neck
        # while preserving the original stock backbone blocks at 6 and 8.
        retained = {i: model.model[i] for i in (6, 8)}
        with torch.random.fork_rng(devices=[]):
            torch.random.default_generator.manual_seed((torch.initial_seed() + 1100) % 2**63)
            apply_baseline(model, "yolo11_dcrcf")
        if not dcn:
            for index, layer in retained.items():
                model.model[index] = layer
    if dcn:
        for index in (6, 8):
            old = retained[index] if rcf else model.model[index]
            with torch.random.fork_rng(devices=[]):
                torch.random.default_generator.manual_seed((torch.initial_seed() + 1200 + index) % 2**63)
                module = ModulatedDCNv2(old.cv1.conv.in_channels, old.cv2.conv.out_channels)
            model.model[index] = graph_layer(module, index, old.f)
        if not rcf:
            model.__class__ = DCNStockLossModel
    model.v4_arm = arm
    assert_graph(model, arm)
    source_state = source.state_dict() if source is not None else {}
    parameter_names = set(dict(model.named_parameters()))
    mapping = {}
    for name, tensor in model.state_dict(keep_vars=True).items():
        retained = original.get(id(tensor))
        source_name = explicit.get(name) if explicit else retained[0] if retained else None
        if source is None or source_name is None:
            continue
        expected = source_state[source_name].to(tensor)
        if not torch.equal(tensor, expected):
            raise ValueError(f"Stage1 tensor mismatch: {name} <- {source_name}")
        mapping[name] = source_name
    model.v4_transfer = {
        "mapping": mapping,
        "transferred_elements": sum(p.numel() for n, p in model.named_parameters() if n in mapping),
        "new_parameters": sorted(parameter_names - mapping.keys()),
        "total_elements": sum(p.numel() for p in model.parameters()),
    }
    return model


def assert_box_loss(criterion, arm):
    if isinstance(criterion.bbox_loss, CIoUNWDBboxLoss) != (arm == "V8F"):
        raise ValueError(f"{arm} has the wrong box loss.")


def graph(model):
    return [
        dict(index=i, source=layer.f, module=type(layer).__name__,
             parameters=sum(p.numel() for p in layer.parameters()))
        for i, layer in enumerate(model.model)
    ]


class MatrixTrainer(PaperBaselineTrainer):
    """Importable trainer. Each DDP worker independently builds the same arm."""

    baseline_id = "gdut_dcrcf_v4"

    def final_eval(self):
        best_epoch = None
        if RANK in {-1, 0} and self.best.is_file():
            checkpoint = torch.load(self.best, map_location="cpu", weights_only=False)
            best_epoch = int(checkpoint["epoch"]) + 1
            del checkpoint
        super().final_eval()
        if best_epoch is not None:
            marker = self.save_dir / "training_complete.json"
            report = json.loads(marker.read_text())
            report["stopper_best_epoch"] = report["best_epoch"]
            report["best_epoch"] = best_epoch
            report["amp_active"] = bool(self.amp)
            report["world_size"] = self.world_size
            report["checkpoint_selection"] = "max_native_validation_mAP50_95_latest_exact_tie"
            marker.write_text(json.dumps(report, indent=2))

    def get_model(self, cfg=None, weights=None, verbose=True):
        arm = os.environ["DCRCF_V4_ARM"]
        track = os.environ.get("DCRCF_V4_TRACK", "gdut")
        if (track == "gdut") != (weights is not None):
            raise ValueError("Checkpoint policy does not match the experiment track.")
        model = DetectionModel(cfg, nc=self.data["nc"], ch=self.data["channels"], verbose=False)
        model = self.set_model_names_for_load(model)
        model = construct(model, arm, weights)
        report = dict(arm=arm, graph=graph(model), inventory=inventory(model),
                      stride=model.stride.tolist(), transfer=model.v4_transfer)
        self.save_dir.mkdir(parents=True, exist_ok=True)
        rank = os.environ.get("RANK", "-1")
        (self.save_dir / f"architecture_rank_{rank}.json").write_text(json.dumps(report, indent=2))
        return model


In [ ]:
%%writefile /kaggle/working/dcrcf_v4_code/v4_paper.py
"""Independent SHWD fold training with an explicit, reviewed input manifest."""

import argparse
from collections import Counter, defaultdict
import json
import os
from pathlib import Path
from statistics import mean, pstdev

from PIL import Image
from hashlib import sha256 as digest
from ultralytics import YOLO
import yaml

from gdut_data import sha256
from v4_data import labels
from v4_model import MatrixTrainer, assert_graph
from v4_run import code_hashes, dump, prepare_validation_model
from warm_environment import runtime_snapshot
from warm_protocol import EVALUATION

NAMES = ["helmet", "person"]
FOLDS = set(range(1, 6))
# The paper gives no exact LR or seed. These values are explicit assumptions.
ARGS = dict(epochs=300, imgsz=640, batch=32, workers=8, optimizer="SGD",
            pretrained=False, resume=False, patience=0, lr0=.01, lrf=.01,
            momentum=.937, weight_decay=.0005, seed=3407, deterministic=True,
            amp=True, cache=False, val=True, save=True, exist_ok=False)


def mounted_registry(input_root):
    candidates = sorted(input_root.rglob("fold_registry.json"))
    if len(candidates) != 1:
        raise ValueError(f"Mount exactly one reviewed SHWD fold registry. Found {len(candidates)}.")
    return candidates[0]


def read_registry(registry):
    if not registry.is_file():
        raise FileNotFoundError("Mount the reviewed SHWD fold registry at " + str(registry))
    source = json.loads(registry.read_text())
    if (source["names"] != NAMES or not source.get("ontology_evidence")
            or not source.get("split_source")
            or source["split_status"] not in {"author_verified", "approximate"}):
        raise ValueError("Missing SHWD split/annotation provenance.")
    root = registry.parent.resolve()
    seen = {field: {} for field in ("pixel", "group")}
    images = set()
    counts = Counter()
    folds = defaultdict(list)
    for row in source["samples"]:
        split = row["split"]
        fold = row["fold"]
        if split not in {"train", "val", "test"} or (
            fold not in FOLDS if split == "train" else fold is not None
        ):
            raise ValueError(f"Invalid fold assignment: {row['image']}")
        img, lab = ((root / row[key]).resolve() for key in ("image", "label"))
        if not (img.is_relative_to(root) and lab.is_relative_to(root)):
            raise ValueError("Files must be under the mounted registry directory.")
        parts = list(img.parts)
        if "images" not in parts:
            raise ValueError(f"Expected standard YOLO images directory: {img}")
        parts[len(parts) - 1 - parts[::-1].index("images")] = "labels"
        if lab != Path(*parts).with_suffix(".txt"):
            raise ValueError(f"Ultralytics would read a different label file: {lab}")
        if img in images or sha256(img) != row["image_sha256"] or sha256(lab) != row["label_sha256"]:
            raise ValueError(f"Duplicate or changed SHWD sample: {img}")
        images.add(img)
        with Image.open(img) as opened:
            pixels = opened.convert("RGB")
            pixel_hash = digest(str(pixels.size).encode() + pixels.tobytes()).hexdigest()
        for field, value in (("pixel", pixel_hash), ("group", row["group"])):
            assignment = split, fold
            if not value or seen[field].setdefault(value, assignment) != assignment:
                raise ValueError(f"Cross-split or cross-fold {field} overlap.")
        classes = {obj[0] for obj in labels(lab)}
        if classes - {0, 1}:
            raise ValueError(f"Unexpected SHWD class: {lab}")
        counts[split] += 1
        if split == "train":
            folds[fold].append((img, classes))
    if source["split_status"] == "author_verified" and counts != {"train": 5306, "val": 1593, "test": 682}:
        raise ValueError("Author-verified registry must match published counts.")
    if set(folds) != FOLDS or any(set.union(*(classes for _, classes in rows)) != {0, 1}
                                   for rows in folds.values()):
        raise ValueError("All five folds must cover both SHWD classes.")
    return source, folds, counts


def run_folds(arm, registry, work):
    source, folds, counts = read_registry(registry)
    work.mkdir(parents=True, exist_ok=False)
    runtime = runtime_snapshot("0", expected_gpu="")
    os.environ.update(DCRCF_V4_TRACK="shwd", DCRCF_V4_ARM=arm)
    for fold in sorted(FOLDS):
        run = work / f"fold-{fold}"
        config = work / f"fold-{fold}.yaml"
        paths = {}
        for split in ("train", "val"):
            listing = work / f"fold-{fold}-{split}.txt"
            selected = [str(img) for i, rows in folds.items() if (i == fold) == (split == "val")
                        for img, _ in rows]
            listing.write_text("\n".join(sorted(selected)) + "\n")
            paths[split] = str(listing)
        config.write_text(yaml.safe_dump(dict(names=NAMES, **paths)))
        trainer = MatrixTrainer(overrides=dict(ARGS, model="yolo11n.yaml", data=str(config),
                                               project=str(work), name=run.name, device="0"))
        trainer.train()
        del trainer
        marker = json.loads((run / "training_complete.json").read_text())
        actual = yaml.safe_load((run / "args.yaml").read_text())
        if marker["epochs_completed"] != 300 or any(actual[k] != v for k, v in ARGS.items()):
            raise ValueError(f"Paper-track recipe or epoch mismatch in fold {fold}")
        if not marker["amp_active"] or marker["world_size"] != 1:
            raise ValueError(f"Paper-track GPU execution mismatch in fold {fold}")
        weight = run / "weights/last.pt"
        model = YOLO(str(weight))
        assert_graph(model.model, arm)
        prepare_validation_model(model)
        result = model.val(data=str(config), split="val", project=str(run), name="epoch300",
                           device="0", save_json=True, **EVALUATION)
        dump(run / "paper_fold_manifest.json", dict(
            arm=arm, fold=fold, checkpoint_epoch=300, checkpoint_sha256=sha256(weight),
            registry_sha256=sha256(registry), split_status=source["split_status"], counts=counts,
            runtime=runtime, code=code_hashes(), args=ARGS,
            precision=float(result.box.mp), recall=float(result.box.mr),
            mAP50=float(result.box.map50), mAP50_95=float(result.box.map),
        ))


def aggregate(stock, dcrcf, work):
    runs = {}
    reference = None
    for arm, root in (("C0", stock), ("C3", dcrcf)):
        for fold in sorted(FOLDS):
            directory = root / f"fold-{fold}"
            row = json.loads((directory / "paper_fold_manifest.json").read_text())
            if (row["arm"], row["fold"], row["checkpoint_epoch"]) != (arm, fold, 300) or (
                sha256(directory / "weights/last.pt") != row["checkpoint_sha256"]
            ):
                raise ValueError(f"Wrong or modified fold artifact: {directory}")
            contract = {k: row[k] for k in ("registry_sha256", "split_status", "args", "code")}
            contract["software"] = row["runtime"]["software"]
            contract["hardware"] = row["runtime"]["hardware"]
            if reference is not None and contract != reference:
                raise ValueError("Paper-track protocol drift.")
            reference = contract
            runs[arm, fold] = row
    work.mkdir(parents=True, exist_ok=False)
    output = {}
    for metric in ("precision", "recall", "mAP50", "mAP50_95"):
        values = {arm: [runs[arm, fold][metric] for fold in sorted(FOLDS)] for arm in ("C0", "C3")}
        values["C3-C0"] = [b-a for a, b in zip(values["C0"], values["C3"])]
        output[metric] = {arm: dict(folds=data, mean=mean(data), population_sd=pstdev(data))
                          for arm, data in values.items()}
    dump(work / "shwd_fold_summary.json", dict(contract=reference, split="fold validation at epoch 300",
                                               results=output))


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--work", required=True, type=Path)
    parser.add_argument("--arm", choices=("C0", "C3"))
    parser.add_argument("--registry", type=Path)
    parser.add_argument("--input-root", type=Path, default=Path("/kaggle/input"))
    parser.add_argument("--stock", type=Path)
    parser.add_argument("--dcrcf", type=Path)
    args = parser.parse_args()
    if args.arm:
        run_folds(args.arm, args.registry or mounted_registry(args.input_root), args.work)
    elif args.stock and args.dcrcf:
        aggregate(args.stock, args.dcrcf, args.work)
    else:
        parser.error("Provide --arm or both --stock and --dcrcf.")


In [ ]:
%%writefile /kaggle/working/dcrcf_v4_code/v4_probe.py
"""Read-only DCNv2 and RAU diagnostics on a fixed validation input."""

import torch
from torch.nn import functional as F

from baseline_modules import ModulatedDCNv2, RecalibrationUnit


def fraction(condition):
    return float(condition.float().mean().item())


def probe(model, tensor):
    """Measure fields on a real validation batch, without changing the model."""
    report, handles = {}, []

    def hook(name):
        def record(module, inputs, output):
            if isinstance(module, ModulatedDCNv2):
                x = inputs[0].float()
                fields = F.conv2d(x, module.offset_mask.weight.float(),
                                  module.offset_mask.bias.float(), padding=1)
                offsets = fields[:, :18].reshape(x.shape[0], 9, 2, *x.shape[-2:])
                masks = fields[:, 18:].sigmoid()
                h, w = x.shape[-2:]
                rows = torch.arange(h, device=x.device)[None, None, :, None]
                cols = torch.arange(w, device=x.device)[None, None, None, :]
                grid = torch.arange(9, device=x.device)[None, :, None, None]
                y = rows + grid // 3 - 1 + offsets[:, :, 0]
                x_sample = cols + grid % 3 - 1 + offsets[:, :, 1]
                report[name] = dict(offset_mean_abs_px=float(offsets.abs().mean()),
                                    offset_max_abs_px=float(offsets.abs().max()),
                                    mask_mean=float(masks.mean()),
                                    mask_saturation=fraction((masks < .01) | (masks > .99)),
                                    sampling_centers_outside_image=fraction(
                                        (y < 0) | (y > h - 1) | (x_sample < 0) | (x_sample > w - 1)
                                    ))
            else:
                a, b = inputs
                ga, gb = module.gate_a(a).sigmoid(), module.gate_b(b).sigmoid()
                direct = (a * ga).abs().mean()
                cross = ((1 - ga) * b * gb).abs().mean()
                report[name] = dict(
                    gate_a_mean=float(ga.mean()), gate_b_mean=float(gb.mean()),
                    gate_a_saturation=fraction((ga < .01) | (ga > .99)),
                    gate_b_saturation=fraction((gb < .01) | (gb > .99)),
                    cross_share_mean_abs=float(cross / (direct + cross).clamp_min(1e-12)))
        return record

    try:
        for name, module in model.named_modules():
            if isinstance(module, (ModulatedDCNv2, RecalibrationUnit)):
                handles.append(module.register_forward_hook(hook(name)))
        if handles:
            with torch.inference_mode():
                model(tensor)
        return report
    finally:
        for handle in handles:
            handle.remove()


In [ ]:
%%writefile /kaggle/working/dcrcf_v4_code/v4_run.py
"""Run one GDUT arm or a bounded CUDA smoke using the same entry point."""

import argparse
import csv
import io
import json
import math
import os
from pathlib import Path
from types import SimpleNamespace

import torch
import ultralytics
import yaml
from torch.nn import functional as F
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionValidator
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import DEFAULT_CFG_DICT
from ultralytics.utils.torch_utils import init_seeds
from torchvision.ops import deform_conv2d

from baseline_modules import ModulatedDCNv2, SBA
from gdut_data import sha256
from v4_data import make_smoke_config, validate_inputs
from v4_model import ARMS, V8_ARMS, MatrixTrainer, assert_box_loss, assert_graph, construct, family_for, graph
from v4_probe import probe
from warm_environment import PIN, fingerprint, runtime_snapshot
from warm_metrics import export_metrics
from warm_protocol import COMMON, EVALUATION

SEEDS = (3407, 3408, 3409, 3410, 3411)
CODE = Path(__file__).resolve().parent


def dump(path, value):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n")
    temporary.replace(path)


def code_hashes():
    return {p.name: sha256(p) for p in sorted(CODE.iterdir()) if p.suffix in {".py", ".txt"}}


def verify_cuda_smoke(family, input_id, runtime):
    slug, folder = (("gdut-dcrcf-v4-smoke", "v4-smoke") if family == "yolo11n"
                    else ("gdut-yolov8-v4-v8smoke", "v4-v8smoke"))
    root = Path("/kaggle/input/notebooks/tundng111") / slug / folder
    receipt = json.loads((root / "CUDA_SMOKE_READY.json").read_text())
    arms = tuple(ARMS) if family == "yolo11n" else V8_ARMS
    if (receipt["family"] != family or receipt["code"] != code_hashes()
            or sorted(receipt["manifests"]) != sorted(arms)):
        raise ValueError("CUDA smoke family, source or arm coverage differs.")
    if sha256(root / receipt["log"]["path"]) != receipt["log"]["sha256"]:
        raise ValueError("CUDA smoke log hash differs.")
    for arm, expected_hash in receipt["manifests"].items():
        path = root / f"{arm}-{SEEDS[0]}" / "run_manifest.json"
        if sha256(path) != expected_hash:
            raise ValueError(f"CUDA smoke manifest hash differs: {arm}")
        row = json.loads(path.read_text())
        if (row["arm"] != arm or row["seed"] != SEEDS[0] or not row["smoke"]
                or row["epochs_completed"] != 1 or row["code"] != receipt["code"]
                or row["execution"] != {"amp_active": True, "world_size": 2}
                or any(row["runtime"][key] != runtime[key] for key in ("hardware", "software"))
                or {k: v for k, v in row["inputs"].items() if k != "smoke_training_list_sha256"} != input_id):
            raise ValueError(f"CUDA smoke execution or mounted input differs: {arm}")
    return sha256(root / "CUDA_SMOKE_READY.json")


def architecture_shapes(model):
    shapes, handles = {}, []
    conv_macs = 0

    def count_macs(module, inputs, output):
        nonlocal conv_macs
        if isinstance(module, torch.nn.Conv2d):
            conv_macs += output.numel() * module.weight[0].numel()
        elif isinstance(module, ModulatedDCNv2):
            spatial = output.shape[0] * output.shape[2] * output.shape[3]
            conv_macs += spatial * (module.weight.numel() + module.offset_mask.weight.numel())

    def record(index):
        def hook(module, inputs, output):
            if isinstance(output, torch.Tensor):
                shapes[index] = list(output.shape)
            elif isinstance(output, tuple) and isinstance(output[0], torch.Tensor):
                shapes[index] = list(output[0].shape)
        return hook
    try:
        for index, layer in enumerate(model.model):
            handles.append(layer.register_forward_hook(record(index)))
        for module in model.modules():
            if isinstance(module, (torch.nn.Conv2d, ModulatedDCNv2)):
                handles.append(module.register_forward_hook(count_macs))
        with torch.inference_mode():
            prediction = model(torch.zeros(1, 3, 640, 640))[0]
        if prediction.shape != (1, 4 + model.model[-1].nc, 8400):
            raise ValueError(f"Unexpected 640px head output: {prediction.shape}")
        return dict(layers=[dict(row, output=shapes[row["index"]]) for row in graph(model)],
                    conv_macs=conv_macs, conv_GFLOPs=2 * conv_macs / 1e9,
                    convention="Conv and deformable kernel/predictor multiply-adds only; excludes sampling, attention matmul, BN, resize and other elementwise ops.")
    finally:
        for handle in handles:
            handle.remove()


def preflight(arm, weight, device):
    init_seeds(SEEDS[0], deterministic=True)
    source = YOLO(str(weight)).model
    model = construct(DetectionModel(f"{family_for(arm)}.yaml", nc=5, verbose=False), arm, source)
    model.args = SimpleNamespace(**DEFAULT_CFG_DICT)
    model.train().to(device)
    image = torch.rand(2, 3, 64, 64, device=device)
    batch = dict(img=image, batch_idx=torch.tensor([0., 1.], device=device),
                 cls=torch.tensor([[0.], [4.]], device=device),
                 bboxes=torch.tensor([[.5, .5, .3, .3], [.4, .6, .2, .3]], device=device))
    with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
        loss, _ = model(batch)
    assert_box_loss(model.criterion, arm)
    loss.sum().backward()
    if not torch.isfinite(loss).all() or any(
        p.grad is None or not torch.isfinite(p.grad).all()
        for p in model.parameters() if p.requires_grad
    ):
        raise ValueError(f"{arm} loss or gradients failed.")
    custom_gradient_norms = {}
    for index, module in enumerate(model.model):
        if isinstance(module, (ModulatedDCNv2, SBA)):
            norm = sum(p.grad.detach().float().norm().item() for p in module.parameters())
            if not math.isfinite(norm) or norm <= 0:
                raise ValueError(f"{arm} layer {index} has zero or nonfinite custom gradients.")
            custom_gradient_norms[index] = norm
    model.eval().cpu()
    model.criterion = None
    with torch.no_grad():
        expected = model(image.cpu())[0]
    buffer = io.BytesIO()
    torch.save(model, buffer)
    buffer.seek(0)
    restored = torch.load(buffer, map_location="cpu", weights_only=False).eval()
    with torch.no_grad():
        torch.testing.assert_close(restored(image.cpu())[0], expected)
    if arm in ARMS and ARMS[arm][0]:
        operator = ModulatedDCNv2(4, 8).to(device)
        x = torch.rand(1, 4, 8, 8, device=device)
        offset = torch.zeros(1, 18, 8, 8, device=device)
        mask = torch.ones(1, 9, 8, 8, device=device)
        torch.testing.assert_close(
            deform_conv2d(x, offset, operator.weight, padding=(1, 1), mask=mask),
            F.conv2d(x, operator.weight, padding=1), atol=1e-5, rtol=1e-4,
        )
    return {"architecture": architecture_shapes(restored), "transfer": model.v4_transfer,
            "custom_gradient_norms": custom_gradient_norms,
            "box_loss": "0.5 CIoU + 0.5 NWD (C=12.8 px)" if arm == "V8F" else "native CIoU"}


class PredictionValidator(DetectionValidator):
    """Persist an explicit empty export when inference returns zero detections."""

    def __call__(self, trainer=None, model=None):
        metrics = super().__call__(trainer=trainer, model=model)
        if trainer is None and self.args.save_json and not self.jdict:
            dump(self.save_dir / "predictions.json", [])
        return metrics

    def preprocess(self, batch):
        batch = super().preprocess(batch)
        if not self.training and self.args.split == "val" and not hasattr(self, "probe_image"):
            self.probe_image = batch["img"][:1].detach().clone()
            self.probe_source = Path(batch["im_file"][0])
        return batch


def prepare_validation_model(model):
    """Fuse eligible layers in FP32 on CPU without running a FLOP profiler."""
    model.model = model.model.cpu().float().eval().fuse(verbose=False)
    # Old C1 checkpoints predate the profiler-safe model class. Ensure that
    # AutoBackend cannot re-enter the stock summary on such a checkpoint.
    if (type(model.model) is DetectionModel and getattr(model.model, "v4_arm", None) == "C1"
            and not model.model.is_fused()):
        raise RuntimeError("Legacy C1 checkpoint would invoke the unsupported THOP profiler.")
    return model


def evaluate(model, config, split, target, device):
    prepare_validation_model(model)
    validators = []

    def capture_validator(*args, **kwargs):
        validator = PredictionValidator(*args, **kwargs)
        validators.append(validator)
        return validator

    result = model.val(validator=capture_validator, data=str(config), split=split, project=str(target.parent),
                       name=target.name, device=device, save_json=True, **EVALUATION)
    if (len(result.nt_per_class) != 5 or any(count <= 0 for count in result.nt_per_class)):
        raise ValueError(f"{split} evaluator did not see every GDUT class.")
    summary, classes, definitions = export_metrics(result)
    dump(target / "metrics.json", dict(metrics=summary, per_class=classes, definitions=definitions))
    predictions = target / "predictions.json"
    if not predictions.is_file():
        raise ValueError(f"Missing prediction export for {split}: {predictions}")
    if split == "val":
        validator, = validators
        dump(target / "module_probe.json", dict(
            validation_image=validator.probe_source.name, image_sha256=sha256(validator.probe_source),
            interpretation="Fixed first validation image; descriptive operator diagnostics, not a causal test.",
            modules=probe(model.model, validator.probe_image),
        ))
    return summary


def train_one(arm, seed, root, config, weight, input_id, runtime, device, smoke):
    os.environ["DCRCF_V4_TRACK"] = "gdut"
    os.environ["DCRCF_V4_ARM"] = arm
    args = dict(COMMON, seed=seed, device=device, model=str(weight), data=str(config),
                project=str(root.parent), name=root.name, exist_ok=False)
    if smoke:
        args.update(epochs=1, plots=False)
    if device == "cpu":
        if not smoke:
            raise ValueError("The matched study requires the pinned CUDA runtime.")
        args.update(imgsz=64, workers=0, batch=2, amp=False, fraction=1.0)
    trainer = MatrixTrainer(overrides=args)
    trainer.train()
    del trainer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    actual = yaml.safe_load((root / "args.yaml").read_text())
    for key in COMMON.keys() | {"data", "device", "model", "seed"}:
        if actual.get(key) != args[key]:
            raise ValueError(f"Executed argument differs: {key} {actual.get(key)!r} != {args[key]!r}")
    marker = json.loads((root / "training_complete.json").read_text())
    expected_world_size = 0 if device == "cpu" else len(device.split(","))
    if marker["amp_active"] != args["amp"] or marker["world_size"] != expected_world_size:
        raise ValueError(f"Actual AMP or world size differs from the run contract: {marker}")
    with (root / "results.csv").open() as stream:
        rows = list(csv.DictReader(stream))
    if (len(rows) != marker["epochs_completed"] or not 1 <= len(rows) <= args["epochs"]
            or [int(row["epoch"]) for row in rows] != list(range(1, len(rows) + 1))
            or not all(math.isfinite(float(value)) for row in rows for value in row.values())):
        raise ValueError("Epoch coverage or nonfinite metric failure.")
    best_epoch = marker["best_epoch"]
    if not isinstance(best_epoch, int) or not 1 <= best_epoch <= len(rows):
        raise ValueError(f"Missing best checkpoint epoch: {best_epoch}")
    checkpoints = {f"weights/{key}.pt": sha256(root / f"weights/{key}.pt") for key in ("best", "last")}
    model = YOLO(str(root / "weights/best.pt"))
    assert_graph(model.model, arm)
    if model.model.v4_arm != arm:
        raise ValueError("Reloaded checkpoint arm differs.")
    expected_ranks = (-1,) if "," not in device else tuple(range(len(device.split(","))))
    ranks = [json.loads((root / f"architecture_rank_{i}.json").read_text()) for i in expected_ranks]
    if any(rank != ranks[0] for rank in ranks) or ranks[0]["graph"] != graph(model.model):
        raise ValueError("DDP rank or saved checkpoint graph mismatch.")
    validation = evaluate(model, config, "val", root / "validation", device.split(",")[0])
    dump(root / "run_manifest.json", dict(
        revision="dcrcf_v4", track="gdut", arm=arm, seed=seed, smoke=smoke,
        inputs=input_id, code=code_hashes(), runtime=runtime, args=actual,
        best_epoch=best_epoch, epochs_completed=len(rows), checkpoint_sha256=checkpoints,
        checkpoint_selection=marker["checkpoint_selection"],
        execution=dict(amp_active=marker["amp_active"], world_size=marker["world_size"]),
        validation=validation, graph=ranks[0],
    ))
    print(f"{arm} seed={seed} epochs={len(rows)} best={best_epoch} val_AP50_95={validation['mAP50_95']:.5f}", flush=True)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--family", choices=("yolo11n", "yolov8s"), default="yolo11n")
    parser.add_argument("--arm", choices=(*ARMS, *V8_ARMS))
    parser.add_argument("--smoke", action="store_true")
    parser.add_argument("--graph-only", action="store_true")
    parser.add_argument("--work", type=Path, required=True)
    args = parser.parse_args()
    if ultralytics.__version__ != PIN:
        raise RuntimeError(f"Expected Ultralytics {PIN}.")
    family_arms = tuple(ARMS) if args.family == "yolo11n" else V8_ARMS
    if (args.arm is not None and args.arm not in family_arms
            or not (args.smoke or args.graph_only) and args.arm is None):
        parser.error(f"Select an arm from {family_arms}.")
    arms = family_arms if args.smoke or args.graph_only else (args.arm,)
    args.work.mkdir(parents=True, exist_ok=False)
    device = "0,1"
    try:
        runtime = runtime_snapshot(device, "Tesla T4")
        config, weight, audit = validate_inputs(args.work, family=args.family)
        dump(args.work / "data_audit.json", audit)
        input_id = {k: audit[k] for k in ("manifest_sha256", "augmentation_manifest_sha256", "stage1_checkpoint_sha256")}
        input_id["train_val_contents"] = fingerprint(audit["train_val_sha256"])
        if not args.smoke and not args.graph_only:
            dump(args.work / "cuda_smoke_verified.json",
                 {"receipt_sha256": verify_cuda_smoke(args.family, input_id, runtime)})
        torch.set_num_threads(2)
        for arm in arms:
            dump(args.work / f"preflight_{arm}.json", preflight(arm, weight, torch.device("cuda:0")))
        torch.cuda.empty_cache()
        if args.graph_only:
            return
        if args.smoke:
            config = make_smoke_config(config, args.work)
            input_id["smoke_training_list_sha256"] = sha256(args.work / "smoke_train_images.txt")
        seeds = SEEDS[:1] if args.smoke else SEEDS
        for arm in arms:
            for seed in seeds:
                train_one(arm, seed, args.work / f"{arm}-{seed}", config, weight,
                          input_id, runtime, device, args.smoke)
        print("V4 smoke passed" if args.smoke else f"{args.arm} five seeds complete", flush=True)
    except BaseException as error:
        dump(args.work / "failure.json", dict(error=type(error).__name__, message=str(error)))
        raise


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/dcrcf_v4_code/prep_dataset_pipeline.py.txt
"""Dataset-specific adapters. Run against Kaggle mounts, never download images."""
from pathlib import Path
import collections
import hashlib
import json
import random
import re
import shutil
import xml.etree.ElementTree as ET
import numpy as np
from PIL import Image

NAMES = {'gdut': ['blue', 'white', 'yellow', 'red', 'none'],
         'iitu': ['blue helmet', 'green helmet', 'red helmet', 'white helmet', 'yellow helmet']}
SEED = 3407

def dump(path, obj):
    Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding='utf-8')

def image_info(path):
    # Decode fully: fail rather than silently losing corrupt examples.
    with Image.open(path) as im:
        im = im.convert('RGB'); im.load()
        return im.size, hashlib.sha256(str(im.size).encode() + im.tobytes()).hexdigest()

def valid_boxes(boxes, path):
    for cls, x, y, w, h in boxes:
        if cls != int(cls) or not 0 <= cls < 5 or not np.isfinite([x,y,w,h]).all():
            raise ValueError(f'Invalid class/nonfinite label: {path}: {cls,x,y,w,h}')
        if w <= 0 or h <= 0 or min(x-w/2,y-h/2) < -1e-6 or max(x+w/2,y+h/2) > 1+1e-6:
            raise ValueError(f'Invalid normalized box: {path}: {cls,x,y,w,h}')
    return boxes

def read_gdut(root):
    ann = root/'Annotations'; imgs = root/'JPEGImages'
    if not imgs.is_dir():
        raise FileNotFoundError(f'Expected VOC JPEGImages: {imgs}')
    trainval = (root/'ImageSets/Main/trainval.txt').read_text().split()
    test = (root/'ImageSets/Main/test.txt').read_text().split()
    assert len(set(trainval)) == len(trainval) and len(set(test)) == len(test)
    assert not set(trainval) & set(test), 'Official split overlap'
    files = sorted(ann.glob('*.xml'))
    assert {p.stem for p in files} == set(trainval+test), 'Unlisted/missing XML'
    records = []
    for index,p in enumerate(files):
        if index%100==0: print(f'[SCAN GDUT] {index}/{len(files)} images decoded/hashed',flush=True)
        t = ET.parse(p).getroot(); image = imgs/(p.stem+'.jpg')
        (w,h), digest = image_info(image)
        assert (w,h) == (int(t.findtext('size/width')), int(t.findtext('size/height'))), p
        boxes=[]
        for obj in t.findall('object'):
            name=obj.findtext('name').strip()
            if name not in NAMES['gdut']: raise ValueError(f'Unknown GDUT class {name}: {p}')
            b=obj.find('bndbox'); x1,y1,x2,y2=[float(b.findtext(k)) for k in ('xmin','ymin','xmax','ymax')]
            # VOC is 1-based inclusive. Permit zero-valued lower edge found in some exports.
            x1=max(0,x1-1); y1=max(0,y1-1); x2=min(w,x2); y2=min(h,y2)
            boxes.append([NAMES['gdut'].index(name),(x1+x2)/(2*w),(y1+y2)/(2*h),(x2-x1)/w,(y2-y1)/h])
        records.append(dict(source=str(image),relative=image.relative_to(root).as_posix(),
                            boxes=valid_boxes(boxes,p),pixel_hash=digest,
                            width=w,height=h,group=p.stem,official='test' if p.stem in set(test) else 'trainval'))
    assert len(records)==3174, 'Dataset version changed; review before training'
    return records

def read_iitu(root):
    records=[]
    for p in sorted(root.rglob('*.txt')):
        if p.parent.name != 'annotations': continue
        image=p.parent.parent/'images'/(p.stem+'.jpg')
        (w,h),digest=image_info(image)
        if len(records)%100==0:print(f'[SCAN IITU] {len(records)}/1664 images decoded/hashed',flush=True)
        rows=[]
        for line in p.read_text().splitlines():
            if not line.strip(): continue
            vals=line.split()
            if len(vals)!=5: raise ValueError(f'Not YOLO detection format: {p}: {line}')
            rows.append(list(map(float,vals)))
        rel=p.relative_to(root).as_posix()
        lighting='night' if 'frames_night_time/' in rel else 'day'
        video=re.sub(r'(?:_?frame_?\d+)$','',p.stem)
        if video==p.stem: raise ValueError(f'Unrecognized video/frame name: {p}')
        # Merge same prefix across angle/distance folders; conservatively merge anonymous night frames.
        group=lighting+'|'+(video or 'anonymous_frames')
        records.append(dict(source=str(image),relative=image.relative_to(root).as_posix(),
                            boxes=valid_boxes(rows,p),pixel_hash=digest,group=group,lighting=lighting,width=w,height=h,official=None))
    assert len(records)==1664, 'Dataset version changed; review before training'
    # Five presentation samples in labeled_samples_h0_3m have no paired labels and are excluded explicitly.
    return records

def split_records(records, key, audit=None):
    rng=random.Random(SEED)
    records=[dict(r) for r in records]
    if key=='gdut':
        test_by_hash=collections.defaultdict(list)
        for r in records:
            if r['official']=='test':test_by_hash[r['pixel_hash']].append(r.get('relative',r['group']))
        excluded=[r for r in records if r['official']=='trainval' and r['pixel_hash'] in test_by_hash]
        if audit is not None:
            audit.update(policy='preserve_all_official_test_remove_overlapping_trainval',
                excluded_trainval=[dict(relative=r.get('relative',r['group']),pixel_hash=r['pixel_hash'],
                    matching_test=test_by_hash[r['pixel_hash']]) for r in excluded],
                official_test_count=sum(r['official']=='test' for r in records))
        records=[r for r in records if not (r['official']=='trainval' and r['pixel_hash'] in test_by_hash)]
        print(f'[DEDUP GDUT] Excluded {len(excluded)} trainval images matching official test; test unchanged.',flush=True)
    # Identical decoded images are one indivisible group, even if filenames differ.
    parent={r['group']:r['group'] for r in records}
    def find(x):
        while parent[x]!=x:
            parent[x]=parent[parent[x]]; x=parent[x]
        return x
    seen={}
    for r in records:
        g=r['group']; old=seen.setdefault(r['pixel_hash'],g)
        parent[find(g)]=find(old)
    for r in records: r['group']=find(r['group'])
    groups=collections.defaultdict(list)
    for i,r in enumerate(records): groups[r['group']].append(i)
    if key=='gdut':
        for ids in groups.values():
            if len({records[i]['official'] for i in ids})>1:
                raise ValueError('Exact image duplicate crosses official GDUT test/trainval; inspect before training.')
        candidates=[g for g,ids in groups.items() if records[ids[0]]['official']=='trainval']
        rng.shuffle(candidates); target=round(sum(len(groups[g]) for g in candidates)*0.2)
        val=set(); n=0
        for g in candidates:
            if n>=target: break
            val.add(g); n+=len(groups[g])
        assignment={g:('test' if records[ids[0]]['official']=='test' else 'val' if g in val else 'train') for g,ids in groups.items()}
    else:
        keys=sorted(groups); assert len(keys)>=3
        counts={g:np.bincount([int(b[0]) for i in groups[g] for b in records[i]['boxes']], minlength=5) for g in keys}
        lights={g:collections.Counter(records[i].get('lighting',records[i]['group'].split('|')[0]) for i in groups[g]) for g in keys}
        # Search group assignments using labels/counts only, never predictions/test performance.
        best=None
        for _ in range(12000):
            rng.shuffle(keys)
            assignment={g:rng.choices(['train','val','test'],[.7,.15,.15])[0] for g in keys}
            sizes=np.array([sum(len(groups[g]) for g in keys if assignment[g]==s) for s in ['train','val','test']])
            class_counts=[sum((counts[g] for g in keys if assignment[g]==s),np.zeros(5)) for s in ['train','val','test']]
            if min(sizes)==0 or any((v==0).any() for v in class_counts): continue
            if any(sum(lights[g][light] for g in keys if assignment[g]==s)<20 for s in ['train','val','test'] for light in ['day','night']):continue
            score=float(np.square(sizes/len(records)-[.7,.15,.15]).sum())
            if best is None or score<best[0]: best=(score,assignment.copy())
        if best is None: raise ValueError('No group split covers all five classes. Review groups; do not use random frame split.')
        assignment=best[1]
    for r in records:r['split']=assignment[r['group']]
    for s in ['train','val','test']:
        classes={int(b[0]) for r in records if r['split']==s for b in r['boxes']}
        assert classes==set(range(5)), f'{s} missing classes: {classes}'
    return records

def prepare(key, root, out, aug_image_size=640):
    root,out=Path(root),Path(out)
    if out.exists(): raise FileExistsError(f'Refusing to mix/rewrite an existing prepared dataset: {out}')
    duplicate_audit={}
    records=split_records(read_gdut(root) if key=='gdut' else read_iitu(root),key,duplicate_audit)
    out.mkdir(parents=True)
    dump(out/'duplicate_audit.json',duplicate_audit if key=='gdut' else {'policy':'merge_exact_duplicates_into_video_groups'})
    for r in records:
        uid=hashlib.sha256(r['relative'].encode()).hexdigest()[:20]
        r['id']=uid
        dst=out/'base/images'/r['split']/(uid+'.jpg'); dst.parent.mkdir(parents=True,exist_ok=True)
        # Copy to writable Kaggle output; leave original image pixels unchanged.
        shutil.copy2(r['source'],dst)
        lab=out/'base/labels'/r['split']/(uid+'.txt'); lab.parent.mkdir(parents=True,exist_ok=True)
        lab.write_text(''.join(f'{int(b[0])} '+' '.join(f'{x:.10f}' for x in b[1:])+'\n' for b in r['boxes']))
    manifest={'dataset':key,'names':NAMES[key],'seed':SEED,'records':records,'preparation_version':2,
              'augmentation_imgsz':aug_image_size,'duplicate_audit':duplicate_audit,
              'split_counts':dict(collections.Counter(r['split'] for r in records)),
              'box_counts':{s:dict(collections.Counter(NAMES[key][int(b[0])] for r in records if r['split']==s for b in r['boxes'])) for s in ['train','val','test']},
              'group_assignment':{r['group']:r['split'] for r in records},
              'notes':'IITU group split unchanged. GDUT test unchanged; overlapping trainval copies excluded and audited BEFORE remaining trainval split 80/20. All retained objects, including VOC difficult, retained.'}
    dump(out/'dataset_manifest.json',manifest)
    build_aug(out,aug_image_size)
    dump(out/'preparation_complete.json',{'complete':True,'records':len(records),'augmentation_imgsz':aug_image_size})
    return manifest

def build_aug(out,image_size=640):
    import cv2
    from albumentations_hardcase_policy import build_hardcase_transform,read_yolo_label,write_yolo_label
    out=Path(out); transform=build_hardcase_transform(image_size)
    # Color is the target in these datasets: remove hue/saturation shift, retain geometry/occlusion/illumination policy.
    transform.transforms=[t for t in transform.transforms if t.__class__.__name__!='HueSaturationValue']
    for i,image in enumerate(sorted((out/'base/images/train').glob('*.jpg'))):
        if i%100==0:print(f'[HARD AUG] {i} images completed',flush=True)
        transform.set_random_seed(SEED+i)
        lab=out/'base/labels/train'/(image.stem+'.txt')
        boxes,classes=read_yolo_label(lab)
        rgb=cv2.cvtColor(cv2.imread(str(image)),cv2.COLOR_BGR2RGB)
        result=transform(image=rgb,bboxes=boxes,class_labels=classes)
        dst=out/'hardaug/images/train'/(image.stem+'_hardaug.jpg');dst.parent.mkdir(parents=True,exist_ok=True)
        assert cv2.imwrite(str(dst),cv2.cvtColor(result['image'],cv2.COLOR_RGB2BGR))
        write_yolo_label(out/'hardaug/labels/train'/(image.stem+'_hardaug.txt'),result['bboxes'],result['class_labels'])
    dump(out/'augmentation_manifest.json',{'seed':SEED,'image_size':image_size,'per_image_seed':'3407 + index in sorted hashed image names',
         'copies_per_train_image':1,'hue_saturation_transform_removed':True,'validation_test_augmented':False})

def mounted_yaml(prepared, augmented=False):
    import yaml
    prepared=Path(prepared); meta=json.loads((prepared/'dataset_manifest.json').read_text())
    work=Path('/kaggle/working/config');work.mkdir(parents=True,exist_ok=True)
    data={'path':str(prepared),'names':dict(enumerate(meta['names'])),
          'train':[str(prepared/'base/images/train')]+([str(prepared/'hardaug/images/train')] if augmented else []),
          'val':str(prepared/'base/images/val'),'test':str(prepared/'base/images/test')}
    path=work/(meta['dataset']+('_aug' if augmented else '_base')+'.yaml')
    path.write_text(yaml.safe_dump(data,allow_unicode=True));return str(path)

def find_prepared(key, explicit=''):
    paths=[Path(explicit)] if explicit else [p.parent for p in Path('/kaggle/input').rglob('dataset_manifest.json')]
    paths=[p for p in paths if (p/'dataset_manifest.json').exists() and json.loads((p/'dataset_manifest.json').read_text())['dataset']==key]
    if len(paths)!=1: raise RuntimeError(f'Set PREPARED_DIR explicitly: found {paths}')
    meta=json.loads((paths[0]/'dataset_manifest.json').read_text())
    if meta.get('preparation_version',1)>=2 and not (paths[0]/'preparation_complete.json').exists():
        raise RuntimeError('Prepared output is incomplete: hard augmentation did not finish. Rerun preparation in a fresh output.')
    return paths[0]


In [ ]:
%%writefile /kaggle/working/dcrcf_v4_code/prep_albumentations_hardcase_policy.py.txt
"""
Albumentations hard-case augmentation policy for Stage 2 experiments.

This file is intentionally separate from the stable baseline pipeline. Stage 1
uses stock Ultralytics augmentation knobs for fair comparison. Stage 2 can use
this policy in a custom Dataset/Trainer when testing hard-case robustness.
"""

from __future__ import annotations

from pathlib import Path
from typing import Iterable

import cv2
import numpy as np


def build_hardcase_transform(image_size: int = 640):
    """
    Build an Albumentations transform for helmet hard cases.

    Targeted effects:
    - RandomShadow: cloud/scaffolding shadows.
    - RandomBrightnessContrast: harsh sunlight and low light.
    - HueSaturationValue: color robustness against yellow buckets/signs.
    - CoarseDropout: partial occlusion from scaffolding/grid structures.

    Bboxes use YOLO format: class_id x_center y_center width height.
    """
    import albumentations as A

    coarse_kwargs = {
        "num_holes_range": (1, 8),
        "hole_height_range": (0.03, 0.18),
        "hole_width_range": (0.03, 0.18),
        "fill": 0,
        "p": 0.35,
    }

    return A.Compose(
        [
            A.LongestMaxSize(max_size=image_size, p=1.0),
            A.PadIfNeeded(min_height=image_size, min_width=image_size, border_mode=cv2.BORDER_CONSTANT, fill=114, p=1.0),
            A.RandomShadow(shadow_roi=(0, 0, 1, 1), num_shadows_limit=(1, 3), shadow_dimension=5, p=0.35),
            A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.45),
            A.HueSaturationValue(hue_shift_limit=18, sat_shift_limit=45, val_shift_limit=35, p=0.55),
            A.MotionBlur(blur_limit=5, p=0.12),
            A.GaussNoise(std_range=(0.02, 0.08), p=0.18),
            A.CoarseDropout(**coarse_kwargs),
        ],
        bbox_params=A.BboxParams(format="yolo", label_fields=["class_labels"], min_visibility=0.20),
    )


def read_yolo_label(label_path: Path) -> tuple[list[list[float]], list[int]]:
    boxes: list[list[float]] = []
    labels: list[int] = []
    if not label_path.exists():
        return boxes, labels
    for line in label_path.read_text(encoding="utf-8").splitlines():
        parts = line.strip().split()
        if len(parts) != 5:
            continue
        labels.append(int(float(parts[0])))
        boxes.append([float(value) for value in parts[1:]])
    return boxes, labels


def write_yolo_label(label_path: Path, boxes: Iterable[Iterable[float]], labels: Iterable[int]) -> None:
    lines = []
    for label, box in zip(labels, boxes):
        x, y, w, h = box
        x = min(1.0, max(0.0, float(x)))
        y = min(1.0, max(0.0, float(y)))
        w = min(1.0, max(0.0, float(w)))
        h = min(1.0, max(0.0, float(h)))
        if w <= 0 or h <= 0:
            continue
        lines.append(f"{int(label)} {x:.6f} {y:.6f} {w:.6f} {h:.6f}")
    label_path.parent.mkdir(parents=True, exist_ok=True)
    label_path.write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")


def augment_one_yolo_sample(
    image_path: Path,
    label_path: Path,
    output_image_path: Path,
    output_label_path: Path,
    image_size: int = 640,
) -> bool:
    image = cv2.imread(str(image_path))
    if image is None:
        return False
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    boxes, labels = read_yolo_label(label_path)
    transform = build_hardcase_transform(image_size=image_size)
    augmented = transform(image=image, bboxes=boxes, class_labels=labels)
    out_image = cv2.cvtColor(augmented["image"], cv2.COLOR_RGB2BGR)
    output_image_path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(output_image_path), out_image)
    write_yolo_label(output_label_path, augmented["bboxes"], augmented["class_labels"])
    return True


if __name__ == "__main__":
    print("Import this module from a Stage 2 custom Dataset/Trainer.")


## Execute and save run artifacts

In [ ]:
%%writefile /kaggle/working/dcrcf_v4_code/matched_v5_run.py
"""Run one matched YOLO11n ablation with an in-notebook CUDA smoke gate.

This wrapper deliberately reuses the audited V4 model construction, data
validation, training, evaluation, and artifact code.  It changes only the
execution gate: the target arm must pass its own real-data one-epoch smoke in
the current Kaggle runtime before the five predeclared full seeds start.
"""

import argparse
import json
from pathlib import Path

import torch
import ultralytics

from gdut_data import sha256
from v4_data import make_smoke_config, validate_inputs
from v4_model import ARMS
from v4_run import SEEDS, code_hashes, dump, preflight, train_one
from warm_environment import PIN, fingerprint, runtime_snapshot


STUDY_REVISION = "dcrcf_v5_matched_ablation"


def _read(path):
    return json.loads(path.read_text())


def _verify_self_smoke(path, arm, input_id, runtime):
    row = _read(path)
    expected_inputs = dict(input_id)
    if (
        row.get("arm") != arm
        or row.get("seed") != SEEDS[0]
        or row.get("smoke") is not True
        or row.get("epochs_completed") != 1
        or row.get("execution") != {"amp_active": True, "world_size": 2}
        or row.get("inputs") != expected_inputs
        or any(row.get("runtime", {}).get(key) != runtime[key] for key in ("hardware", "software"))
    ):
        raise RuntimeError("The in-notebook CUDA smoke receipt does not match the full-run contract.")
    return {
        "status": "PASS",
        "study_revision": STUDY_REVISION,
        "arm": arm,
        "seed": SEEDS[0],
        "manifest": path.relative_to(path.parents[1]).as_posix(),
        "manifest_sha256": sha256(path),
        "validation": row["validation"],
        "execution": row["execution"],
        "runtime": {key: row["runtime"][key] for key in ("hardware", "software")},
        "code": code_hashes(),
    }


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--arm", choices=tuple(ARMS), required=True)
    parser.add_argument("--work", type=Path, required=True)
    args = parser.parse_args()

    if ultralytics.__version__ != PIN:
        raise RuntimeError(f"Expected Ultralytics {PIN}, got {ultralytics.__version__}.")

    args.work.mkdir(parents=True, exist_ok=False)
    device = "0,1"
    try:
        runtime = runtime_snapshot(device, "Tesla T4")
        config, weight, audit = validate_inputs(args.work, family="yolo11n")
        dump(args.work / "data_audit.json", audit)
        input_id = {
            key: audit[key]
            for key in ("manifest_sha256", "augmentation_manifest_sha256", "stage1_checkpoint_sha256")
        }
        input_id["train_val_contents"] = fingerprint(audit["train_val_sha256"])

        torch.set_num_threads(2)
        dump(args.work / f"preflight_{args.arm}.json", preflight(args.arm, weight, torch.device("cuda:0")))
        torch.cuda.empty_cache()

        smoke_config = make_smoke_config(config, args.work)
        smoke_inputs = dict(input_id)
        smoke_inputs["smoke_training_list_sha256"] = sha256(args.work / "smoke_train_images.txt")
        smoke_root = args.work / f"self-smoke-{args.arm}-{SEEDS[0]}"
        train_one(args.arm, SEEDS[0], smoke_root, smoke_config, weight,
                  smoke_inputs, runtime, device, smoke=True)
        smoke_receipt = _verify_self_smoke(smoke_root / "run_manifest.json", args.arm,
                                           smoke_inputs, runtime)
        dump(args.work / "SELF_SMOKE_READY.json", smoke_receipt)

        for seed in SEEDS:
            run_root = args.work / f"{args.arm}-{seed}"
            train_one(args.arm, seed, run_root, config, weight,
                      input_id, runtime, device, smoke=False)
            manifest_path = run_root / "run_manifest.json"
            manifest = _read(manifest_path)
            manifest["study_revision"] = STUDY_REVISION
            manifest["self_smoke_receipt_sha256"] = sha256(args.work / "SELF_SMOKE_READY.json")
            dump(manifest_path, manifest)

        dump(args.work / "FULL_RUN_READY.json", {
            "status": "PASS",
            "study_revision": STUDY_REVISION,
            "arm": args.arm,
            "seeds": list(SEEDS),
            "self_smoke_receipt_sha256": sha256(args.work / "SELF_SMOKE_READY.json"),
            "manifests": {
                str(seed): sha256(args.work / f"{args.arm}-{seed}" / "run_manifest.json")
                for seed in SEEDS
            },
        })
        print(f"{args.arm} self-smoke plus five matched seeds complete", flush=True)
    except BaseException as error:
        dump(args.work / "failure.json", {
            "status": "FAIL",
            "study_revision": STUDY_REVISION,
            "arm": args.arm,
            "error": type(error).__name__,
            "message": str(error),
        })
        raise


if __name__ == "__main__":
    main()


In [ ]:
import json
from gdut_data import sha256
from warm_launch import run_logged
from v4_run import SEEDS, dump

WORK = Path("/kaggle/working/v5-matched-c3")
log_path = Path("/kaggle/working/v5-matched-c3.log")
command = [sys.executable, str(CODE / "matched_v5_run.py"),
           "--arm", "C3", "--work", str(WORK)]
try:
    run_logged(command, log_path)
except BaseException:
    if WORK.is_dir() and log_path.is_file():
        log_path.rename(WORK / log_path.name)
    raise
log_path = log_path.rename(WORK / log_path.name)
artifacts = {
    path.relative_to(WORK).as_posix(): sha256(path)
    for path in sorted(WORK.rglob("*"))
    if path.is_file()
}
dump(WORK / "ADD_INPUT_READY.json", {
    "status": "ADD_INPUT_READY",
    "study_revision": "dcrcf_v5_matched_ablation",
    "arm": "C3",
    "seeds": list(SEEDS),
    "full_log": {"path": log_path.name, "sha256": sha256(log_path)},
    "artifacts": artifacts,
})
print(f"Full log: {log_path.name} · SHA-256 {sha256(log_path)}")
print(WORK / "SELF_SMOKE_READY.json")
print(WORK / "FULL_RUN_READY.json")
print(WORK / "ADD_INPUT_READY.json")

## Saved output

In [ ]:
for manifest in sorted(WORK.glob("*/run_manifest.json")):
    row = json.loads(manifest.read_text())
    print(f"{row['arm']} · seed {row['seed']} · {row['epochs_completed']} epochs · "
          f"best epoch {row['best_epoch']} · val mAP50–95 {row['validation']['mAP50_95']:.4f}")
for name in ("CUDA_SMOKE_READY.json", "ADD_INPUT_READY.json", "paired_summary.json", "shwd_fold_summary.json"):
    if (WORK / name).is_file():
        print(WORK / name)
print(f"Saved artifacts: {WORK}")